# Higgsfield — Expansion, Journey, and Segments

Companion to `higgsfield_drivers_1_revenue_retention.ipynb`. Shared rules: clock starts at first charge; first renewal is ~30 days; expansion is upgrade vs credits, not one blob.

This notebook:

1. **§3 Expansion** — plan ladder, upgrade vs credits, timing, and headroom.
2. **CJM** — nested journey gates from paid → habit → renewal, and the high-usage fork.
3. **§5 Hypothesis tests → segments** — leak-free first-28d features, bivariate tests, CatBoost/SHAP, then the five behavioural segments.


## 0. Setup

In [1]:
import pandas as pd
import numpy as np
import nbformat  # required for plotly fig.show() mime rendering in notebooks
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io._renderers as _plotly_renderers
from _plotly_utils.optional_imports import _not_importable

# Plotly caches failed optional imports for the kernel lifetime; clear + rebind.
_not_importable.discard('nbformat')
_plotly_renderers.nbformat = nbformat

pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 220)
pd.set_option('display.float_format', '{:,.2f}'.format)

PALETTE = ['#6C5CE7', '#00B894', '#FDCB6E', '#E17055', '#0984E3', '#2D3436', '#B2BEC3']
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = PALETTE
px.defaults.height = 420

PATH = 'data/'  # folder with the csv files

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.decomposition import FactorAnalysis, PCA
from sklearn.preprocessing import StandardScaler
from scipy.stats import chi2_contingency, mannwhitneyu, kruskal, spearmanr

from catboost import CatBoostClassifier, Pool
import shap

/Users/polina/Desktop/test assignment/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
generations_df = pd.read_csv(PATH + 'generations.csv')
customers_df = pd.read_csv(PATH + 'customers.csv')
charges_df = pd.read_csv(PATH + 'charges.csv')

def to_utc(s):
    return pd.to_datetime(s.astype(str).str.removesuffix(' UTC'), format='ISO8601', utc=True, errors='coerce')

for c in ['in_progress_at', 'completed_at', 'failed_at']:
    generations_df[c] = to_utc(generations_df[c])
customers_df['created_at'] = to_utc(customers_df['created_at'])
charges_df['revenue_time'] = to_utc(charges_df['revenue_time'])

OUTLIERS = list(generations_df['user_id'].value_counts().index[:1])
drop_cust = set(customers_df.loc[customers_df['user_id'].isin(OUTLIERS), 'customer'].dropna())
generations_df = generations_df[~generations_df['user_id'].isin(OUTLIERS)].copy()
customers_df = customers_df[~customers_df['user_id'].isin(OUTLIERS)].copy()
charges_df = charges_df[~charges_df['customer'].isin(drop_cust)].copy()

generations_df['failed'] = generations_df['failed_at'].notna() & generations_df['completed_at'].isna()
generations_df['is_video'] = (generations_df['type'] == 'video').astype(int)
charges_df['is_credits'] = charges_df['subscription_plan'] == 'Credits Package'

DATA_END = max(generations_df['in_progress_at'].max(), charges_df['revenue_time'].max())
DATA_END_BILLING = charges_df['revenue_time'].max()

print(f'generations {len(generations_df):,} | customers {len(customers_df):,} | charges {len(charges_df):,}')
print(f'data ends {DATA_END} | last charge {DATA_END_BILLING} | gap {(DATA_END - DATA_END_BILLING).days}d')

generations 4,949,426 | customers 19,534 | charges 47,291
data ends 2025-10-01 00:13:14.998436+00:00 | last charge 2025-09-30 23:53:39+00:00 | gap 0d


In [4]:
# same charge typing and customer clock as notebook 1
ch = charges_df.sort_values(['customer', 'revenue_time']).copy()
ch['seq'] = ch.groupby('customer').cumcount()
ch['prev_time'] = ch.groupby('customer')['revenue_time'].shift()
ch['gap_days'] = (ch['revenue_time'] - ch['prev_time']).dt.total_seconds() / 86400

rank = {'Higgsfield Basic': 1, 'Higgsfield Pro': 2, 'Higgsfield Ultimate': 3, 'Higgsfield Creator': 4}
sub_only = ch[~ch['is_credits']].copy()
sub_only['prev_plan'] = sub_only.groupby('customer')['subscription_plan'].shift()
plan_dir = sub_only.set_index('charge_id').apply(
    lambda r: np.sign(rank.get(r['subscription_plan'], 0) - rank.get(r['prev_plan'], 0))
    if pd.notna(r['prev_plan']) else 0, axis=1)
ch['plan_dir'] = ch['charge_id'].map(plan_dir).fillna(0)

period_days = ch['subscription_period'].map({'month': 30, 'year': 365})
ch['rev_type'] = np.select(
    [ch['seq'] == 0,
     ch['is_credits'] | (ch['plan_dir'] > 0),
     (ch['payment_type'] == 'Reactivation') | (ch['gap_days'] > period_days.fillna(30) + 15)],
    ['new', 'expansion', 'recovery'], default='retained')

GRACE = 7
PERIOD = 30

sub = charges_df[~charges_df['is_credits']].merge(
    charges_df.groupby('customer')['revenue_time'].min().rename('first_pay'), on='customer')
sub['m'] = ((sub['revenue_time'] - sub['first_pay']).dt.total_seconds() // (PERIOD * 86400)).astype(int)

base = customers_df.merge(
    charges_df.groupby('customer').agg(
        first_pay=('revenue_time', 'min'),
        last_pay=('revenue_time', 'max'),
        n_charges=('charge_id', 'size'),
        revenue=('sales_amount', 'sum'),
        first_plan=('subscription_plan', 'first'),
        first_period=('subscription_period', 'first'),
    ), left_on='customer', right_index=True, how='inner')

base['tenure_days'] = (DATA_END_BILLING - base['first_pay']).dt.total_seconds() / 86400
base['cohort'] = base['first_pay'].dt.tz_localize(None).dt.to_period('M').astype(str)
print(f'charges {len(ch):,} | base {len(base):,} | monthly {(base["first_period"]=="month").sum():,}')
print(ch['rev_type'].value_counts().to_string())


charges 47,291 | base 19,532 | monthly 17,341
rev_type
new          19532
retained     16390
expansion    10071
recovery      1298


# 3. Expansion


We consider the plan logic as Basic < Pro < Ultimate < Creator based on pricing difference, consistent over regions. 

In [5]:
# prove the ladder: Basic < Pro < Ultimate < Creator on price, and that order holds by region
LADDER = ['Higgsfield Basic', 'Higgsfield Pro', 'Higgsfield Ultimate', 'Higgsfield Creator']
short = {p: p.replace('Higgsfield ', '') for p in LADDER}

# new monthly subscriptions = clearest list-price signal (exclude credits / updates / annual noise)
create = charges_df[
    (~charges_df['is_credits'])
    & (charges_df['payment_type'] == 'Subscription Create')
    & (charges_df['subscription_period'] == 'month')
    & (charges_df['subscription_plan'].isin(LADDER))
].copy()
create = create.merge(customers_df[['customer', 'region']], on='customer', how='left')

# modal price per plan (= USD list price proxy); everything else is FX / discounts
list_price = (create.groupby('subscription_plan')['sales_amount']
              .agg(lambda s: s.mode().iat[0])
              .reindex(LADDER))
print('modal monthly create price by plan (ladder order):')
print(list_price.round(2).to_string())
print(f'\nstrictly increasing: {list_price.is_monotonic_increasing}')

# region x plan: median paid amount (FX-aware); order should still be Basic < ... < Creator
reg = (create.groupby(['region', 'subscription_plan'])['sales_amount']
       .median().unstack('subscription_plan').reindex(columns=LADDER))
print(f'\nmedian create price by region x plan:\n{reg.round(2).to_string()}')

order_ok = reg.apply(lambda r: r.dropna().is_monotonic_increasing, axis=1)
print(f'\nregions where plan order is preserved: {order_ok.sum()}/{order_ok.notna().sum()}')
if (~order_ok).any():
    print('violations:\n', order_ok[~order_ok])

# plot 1 — overall ladder
lp = list_price.rename(short).reset_index()
lp.columns = ['plan', 'modal_price']
lp['plan'] = pd.Categorical(lp['plan'], list(short.values()), ordered=True)
fig = px.bar(lp, x='plan', y='modal_price', text='modal_price',
             title='plan ladder by modal monthly create price')
fig.update_traces(texttemplate='$%{text:.0f}', textposition='outside')
fig.update_layout(yaxis_title='USD (modal)', showlegend=False)
fig.show()

# plot 2 — same ladder by region
long = reg.reset_index().melt(id_vars='region', var_name='plan', value_name='median_price')
long = long.dropna(subset=['median_price'])
long['plan'] = long['plan'].map(short)
long['plan'] = pd.Categorical(long['plan'], list(short.values()), ordered=True)
fig = px.line(long, x='plan', y='median_price', color='region', markers=True,
              title='median monthly create price by region — order should stay Basic < Pro < Ultimate < Creator')
fig.update_yaxes(title='median paid, $')
fig.show()

# plot 3 — distribution check (box), still ordered
box = create.copy()
box['plan'] = pd.Categorical(box['subscription_plan'].map(short), list(short.values()), ordered=True)
fig = px.box(box, x='plan', y='sales_amount', color='region',
             title='paid amount at Subscription Create (monthly) — ladder holds inside each region')
fig.update_yaxes(title='sales_amount, $', range=[0, box['sales_amount'].quantile(0.99) * 1.05])
fig.show()


modal monthly create price by plan (ladder order):
subscription_plan
Higgsfield Basic       14.00
Higgsfield Pro         42.89
Higgsfield Ultimate    71.78
Higgsfield Creator    360.67

strictly increasing: True

median create price by region x plan:
subscription_plan  Higgsfield Basic  Higgsfield Pro  Higgsfield Ultimate  Higgsfield Creator
region                                                                                      
T1                            14.00           42.89                71.78              360.67
USA                           14.00           42.89                71.78              360.67
WW                            14.00           42.89                71.78              360.67

regions where plan order is preserved: 3/3


 Let's define expansion as
 1. increase in subscription plan (where 'first_plan' is the original plan paid for by user)
 2. credits top-up. Considered here as non-dependent on any subscription plan, therefore is another part of expansion

Share of each type of expansion month over month

In [6]:
# expansion = plan upgrades (vs original first_plan) + credits top-ups
LADDER = ['Higgsfield Basic', 'Higgsfield Pro', 'Higgsfield Ultimate', 'Higgsfield Creator']
RANK = {p: i + 1 for i, p in enumerate(LADDER)}

first_plan = (ch[~ch['is_credits']].sort_values(['customer', 'revenue_time'])
              .groupby('customer')['subscription_plan'].first())

exp = ch[ch['is_credits'] | (ch['plan_dir'] > 0)].copy()
exp['first_plan'] = exp['customer'].map(first_plan)
exp['exp_type'] = np.where(exp['is_credits'], 'credits', 'upgrade')

up = exp[exp['exp_type'] == 'upgrade'].copy()
up['from_first'] = up['first_plan'].map(RANK)
up['to_rank'] = up['subscription_plan'].map(RANK)
print(f'expansion charges: {len(exp):,} | upgrades: {(exp["exp_type"]=="upgrade").sum():,} | '
      f'credits: {(exp["exp_type"]=="credits").sum():,}')
print(f'upgrades with plan > first_plan: {(up["to_rank"] > up["from_first"]).mean():.1%}')
print(f'\nrevenue by expansion type:\n'
      f'{exp.groupby("exp_type")["sales_amount"].agg(["count","sum","mean"]).round(2)}')

# month-over-month share within expansion
exp['ym'] = exp['revenue_time'].dt.tz_localize(None).dt.to_period('M').astype(str)
by = (exp.groupby(['ym', 'exp_type'])['sales_amount'].sum()
      .rename('revenue').reset_index())
pivot = by.pivot(index='ym', columns='exp_type', values='revenue').fillna(0)
pivot = pivot.sort_index()
if len(pivot) > 1:
    print(f'dropping last month from mix: {pivot.index[-1]}')
    pivot = pivot.iloc[:-1]
months = list(pivot.index)
pct = pivot.div(pivot.sum(axis=1), axis=0)
print(f'\nexpansion mix by month:\n{pct.round(3)}')

pct_long = pct.reset_index().melt(id_vars='ym', var_name='exp_type', value_name='share')
fig = px.area(pct_long, x='ym', y='share', color='exp_type',
              category_orders={'ym': months},
              title='share of expansion revenue: upgrades vs credits, MoM')
fig.update_xaxes(title='month', type='category', categoryorder='array', categoryarray=months,
                 tickmode='array', tickvals=months, ticktext=months)
fig.update_yaxes(tickformat='.0%', title='share of expansion')
fig.show()

by_plot = by[by['ym'].isin(months)].copy()
by_plot['ym'] = pd.Categorical(by_plot['ym'], categories=months, ordered=True)
fig = px.bar(by_plot.sort_values('ym'), x='ym', y='revenue', color='exp_type',
             category_orders={'ym': months},
             title='expansion revenue by type, monthly')
fig.update_xaxes(title='month', type='category', categoryorder='array', categoryarray=months,
                 tickmode='array', tickvals=months, ticktext=months)
fig.show()

# upgrade expansion mix by original first_plan
up['ym'] = up['revenue_time'].dt.tz_localize(None).dt.to_period('M').astype(str)
up_m = (up.groupby(['ym', 'first_plan'])['sales_amount'].sum()
        .rename('revenue').reset_index())
up_m = up_m[up_m['ym'].isin(months)]
up_piv = up_m.pivot(index='ym', columns='first_plan', values='revenue').fillna(0)
up_piv = up_piv.reindex(months)
up_piv = up_piv[[c for c in LADDER if c in up_piv.columns]]
up_pct = up_piv.div(up_piv.sum(axis=1).replace(0, np.nan), axis=0)
up_long = up_pct.reset_index().melt(id_vars='ym', var_name='first_plan', value_name='share')
fig = px.area(up_long, x='ym', y='share', color='first_plan',
              category_orders={'ym': months, 'first_plan': LADDER},
              title='upgrade expansion mix by original first_plan, MoM')
fig.update_xaxes(title='month', type='category', categoryorder='array', categoryarray=months,
                 tickmode='array', tickvals=months, ticktext=months)
fig.update_yaxes(tickformat='.0%')
fig.show()


expansion charges: 10,079 | upgrades: 3,306 | credits: 6,773
upgrades with plan > first_plan: 98.1%

revenue by expansion type:
          count        sum  mean
exp_type                        
credits    6773 194,794.23 28.76
upgrade    3306 210,775.88 63.76
dropping last month from mix: 2025-09

expansion mix by month:
exp_type  credits  upgrade
ym                        
2025-04      0.56     0.44
2025-05      0.59     0.41
2025-06      0.61     0.39
2025-07      0.51     0.49
2025-08      0.42     0.58


In [7]:
# customer-level expansion frame used by the ladder / timing cells
LADDER = ['Higgsfield Basic', 'Higgsfield Pro', 'Higgsfield Ultimate', 'Higgsfield Creator']
RANK = {p: i + 1 for i, p in enumerate(LADDER)}

exp_all = ch[ch['is_credits'] | (ch['plan_dir'] > 0)].copy()
exp_all = exp_all.merge(base[['customer', 'first_pay']], on='customer', how='left')
exp_all['days_in'] = (exp_all['revenue_time'] - exp_all['first_pay']).dt.total_seconds() / 86400
exp_all['kind'] = np.where(exp_all['is_credits'], 'credits', 'upgrade')

exp_base = base.set_index('customer')

xfeat = base.set_index('customer')[['first_plan', 'first_pay', 'tenure_days']].copy()
up_cust = set(exp_all.loc[exp_all['kind'] == 'upgrade', 'customer'])
cr_cust = set(exp_all.loc[exp_all['kind'] == 'credits', 'customer'])
xfeat['upgraded'] = xfeat.index.isin(up_cust).astype(int)
xfeat['bought_credits'] = xfeat.index.isin(cr_cust).astype(int)
xfeat['expanded'] = ((xfeat['upgraded'] == 1) | (xfeat['bought_credits'] == 1)).astype(int)
print(f'expansion events {len(exp_all):,} | upgraded {xfeat["upgraded"].sum():,} | '
      f'credits {xfeat["bought_credits"].sum():,} | either {xfeat["expanded"].sum():,}')


expansion events 10,079 | upgraded 2,878 | credits 2,487 | either 4,435


In [8]:
# when in the journey does each expansion type land — by original first_plan?
LADDER = ['Higgsfield Basic', 'Higgsfield Pro', 'Higgsfield Ultimate', 'Higgsfield Creator']
journey = exp_all.merge(base[['customer', 'first_plan']], on='customer', how='left')
journey = journey[journey['first_plan'].isin(LADDER)].copy()
journey['first_plan'] = pd.Categorical(journey['first_plan'], LADDER, ordered=True)

# first event of each kind per customer (journey clock = days since first_pay)
first_kind = (journey.sort_values('revenue_time')
              .groupby(['customer', 'kind'], as_index=False)
              .first())
print('days to first expansion event by type x first_plan (median):')
print(first_kind.groupby(['first_plan', 'kind'], observed=True)['days_in']
      .agg(['count', 'median', 'mean']).round(1).to_string())

# 1) distribution of timing (first 180d)
fig = px.box(
    first_kind[first_kind['days_in'].between(0, 180)],
    x='first_plan', y='days_in', color='kind',
    title='days to first upgrade / credits by original plan (≤180d)')
fig.update_yaxes(title='days since first payment')
fig.update_xaxes(title='first_plan')
fig.show()

fig = px.violin(
    first_kind[first_kind['days_in'].between(0, 180)],
    x='kind', y='days_in', color='first_plan',
    category_orders={'first_plan': LADDER, 'kind': ['upgrade', 'credits']},
    title='timing density: when each expansion type happens, by first_plan')
fig.update_yaxes(title='days since first payment')
fig.show()

# 2) cumulative share of expanders who have already expanded by day t (among those who ever do)
cdf_rows = []
for (plan, kind), g in first_kind.groupby(['first_plan', 'kind'], observed=True):
    if len(g) < 30:
        continue
    days = np.sort(g['days_in'].clip(lower=0).values)
    grid = np.arange(0, 181, 5)
    for t in grid:
        cdf_rows.append({'first_plan': plan, 'kind': kind, 'day': t,
                         'cdf': (days <= t).mean()})
cdf = pd.DataFrame(cdf_rows)
fig = px.line(cdf, x='day', y='cdf', color='first_plan', facet_col='kind',
              category_orders={'first_plan': LADDER, 'kind': ['upgrade', 'credits']},
              title='among those who expand: cumulative timing of first event')
fig.update_yaxes(tickformat='.0%', title='share already expanded')
fig.update_xaxes(title='days since first payment')
fig.show()

# 3) monthly hazard of FIRST upgrade / FIRST credits, by first_plan
#    at risk = tenure long enough, and not yet had that event type
haz_rows = []
for kind in ['upgrade', 'credits']:
    fk = first_kind[first_kind['kind'] == kind].set_index('customer')['days_in']
    for plan in LADDER:
        base_p = exp_base[exp_base['first_plan'] == plan]
        if len(base_p) < 50:
            continue
        for k in range(0, 6):
            lo, hi = k * 30, (k + 1) * 30
            at_risk = base_p[base_p['tenure_days'] >= hi]
            prior = set(fk[fk < lo].index)
            at_risk = at_risk[~at_risk.index.isin(prior)]
            if len(at_risk) < 30:
                continue
            hit = set(fk[fk.between(lo, hi)].index)
            haz_rows.append({
                'month': k + 1, 'kind': kind, 'first_plan': plan,
                'at_risk': len(at_risk),
                'hazard': at_risk.index.isin(hit).mean(),
            })
haz_p = pd.DataFrame(haz_rows)
print('\nfirst-event hazard by tenure month x plan x type:')
print(haz_p.pivot_table(index=['kind', 'month'], columns='first_plan',
                        values='hazard').round(3).to_string())

fig = px.line(haz_p, x='month', y='hazard', color='first_plan', facet_col='kind',
              markers=True, category_orders={'first_plan': LADDER, 'kind': ['upgrade', 'credits']},
              title='first-event hazard by journey month: who expands this month among those still at risk')
fig.update_xaxes(title='month since first payment', tickmode='array',
                 tickvals=list(range(1, 7)), range=[0.5, 6.5])
fig.update_yaxes(tickformat='.1%', title='hazard')
fig.show()

# 4) share of each type's events landing in early vs later journey windows
journey['window'] = pd.cut(
    journey['days_in'],
    bins=[-0.1, 7, 30, 60, 90, 180, np.inf],
    labels=['0-7d', '8-30d', '31-60d', '61-90d', '91-180d', '180d+'])
win = (journey.groupby(['first_plan', 'kind', 'window'], observed=True)['sales_amount']
       .sum().rename('revenue').reset_index())
win_pct = (win.pivot_table(index=['first_plan', 'window'], columns='kind',
                           values='revenue', aggfunc='sum').fillna(0))
# event counts share within kind x plan
cnt = (journey.groupby(['first_plan', 'kind', 'window'], observed=True)
       .size().rename('n').reset_index())
cnt['share'] = cnt.groupby(['first_plan', 'kind'], observed=True)['n'].transform(lambda s: s / s.sum())
fig = px.bar(cnt, x='window', y='share', color='kind', facet_col='first_plan',
             barmode='group', category_orders={
                 'first_plan': LADDER, 'kind': ['upgrade', 'credits'],
                 'window': ['0-7d', '8-30d', '31-60d', '61-90d', '91-180d', '180d+']},
             title='where in the journey expansion events fall (share of events), by first_plan')
fig.update_yaxes(tickformat='.0%')
fig.show()


days to first expansion event by type x first_plan (median):
                             count  median  mean
first_plan          kind                        
Higgsfield Basic    credits    276    6.10 15.80
                    upgrade    350    0.00 33.40
Higgsfield Pro      credits    581   13.90 26.50
                    upgrade   1404    7.90 24.10
Higgsfield Ultimate credits    483   18.00 29.70
                    upgrade   1019   14.90 28.40
Higgsfield Creator  credits     18   17.50 23.40
                    upgrade     45   20.90 36.90



first-event hazard by tenure month x plan x type:
first_plan     Higgsfield Basic  Higgsfield Creator  Higgsfield Pro  Higgsfield Ultimate
kind    month                                                                           
credits 1                  0.03                0.15            0.06                 0.16
        2                  0.00                 NaN            0.01                 0.06
        3                  0.00                 NaN            0.01                 0.05
        4                  0.00                 NaN            0.01                 0.08
        5                  0.01                 NaN            0.02                 0.08
upgrade 1                  0.02                0.36            0.15                 0.34
        2                  0.01                 NaN            0.03                 0.14
        3                  0.01                 NaN            0.03                 0.14
        4                  0.01                 NaN        

In [9]:
# the plan ladder is ORDERED: Basic < Pro < Ultimate < Creator.
# treating first_plan as a flat category hides the thing that actually drives upgrades -
# how many rungs are left above you. Creator customers cannot upgrade at all, so putting
# them in an upgrade-rate denominator measures the ladder, not the customer.
LADDER = ['Higgsfield Basic', 'Higgsfield Pro', 'Higgsfield Ultimate', 'Higgsfield Creator']
RANK = {p: i + 1 for i, p in enumerate(LADDER)}
TOP = len(LADDER)

xfeat['plan_rank'] = xfeat['first_plan'].map(RANK)
xfeat['rungs_above'] = TOP - xfeat['plan_rank']
xfeat['can_upgrade'] = (xfeat['rungs_above'] > 0).astype(int)

print(f'plan ladder: {" < ".join(p.replace("Higgsfield ", "") for p in LADDER)}\n')
lad = xfeat.groupby(['plan_rank', 'first_plan']).agg(
    n=('expanded', 'size'),
    rungs_above=('rungs_above', 'first'),
    upgrade_rate=('upgraded', 'mean'),
    credits_rate=('bought_credits', 'mean'),
    expanded=('expanded', 'mean')).reset_index().sort_values('plan_rank')
print(lad.round(3).to_string(index=False))

print(f'\ncustomers with no rung above them: {(xfeat["can_upgrade"] == 0).sum():,} '
      f'({(xfeat["can_upgrade"] == 0).mean():.1%}) - they can only buy credits')

plan ladder: Basic < Pro < Ultimate < Creator

 plan_rank          first_plan    n  rungs_above  upgrade_rate  credits_rate  expanded
      1.00    Higgsfield Basic 9419         3.00          0.04          0.03      0.06
      2.00      Higgsfield Pro 6825         2.00          0.21          0.09      0.23
      3.00 Higgsfield Ultimate 2081         1.00          0.49          0.23      0.53
      4.00  Higgsfield Creator   78         0.00          0.58          0.23      0.58

customers with no rung above them: 1,207 (6.2%) - they can only buy credits


In [10]:
# upgrade rate CONDITIONAL on having somewhere to go
elig_up = xfeat[xfeat['can_upgrade'] == 1]
print(f'upgrade rate, all customers:        {xfeat["upgraded"].mean():.1%}')
print(f'upgrade rate, only those who can:   {elig_up["upgraded"].mean():.1%}\n')

cond = elig_up.groupby(['plan_rank', 'first_plan']).agg(
    n=('upgraded', 'size'), rungs=('rungs_above', 'first'),
    upgrade_rate=('upgraded', 'mean')).reset_index().sort_values('plan_rank')
cond = cond[cond['n'] >= 30]
print(f'upgrade rate among customers with headroom:\n{cond.round(3).to_string(index=False)}')

# with only 3-4 rungs a correlation has no power - report the raw pattern instead
print(f'\nrungs vs upgrade rate ({len(cond)} rungs - too few for a meaningful correlation):')
for _, r in cond.iterrows():
    print(f'  {r["first_plan"]:<22} {int(r["rungs"])} rungs above -> {r["upgrade_rate"]:.1%}')
if len(cond) >= 2:
    mono = cond['upgrade_rate'].is_monotonic_decreasing or cond['upgrade_rate'].is_monotonic_increasing
    print(f'-> {"monotone in headroom: likely the LADDER talking, not the customer" if mono else "NOT monotone in headroom: there is a plan effect beyond position"}')

fig = make_subplots(specs=[[{'secondary_y': True}]])
fig.add_bar(x=cond['first_plan'], y=cond['upgrade_rate'], name='upgrade rate',
            marker_color=PALETTE[0])
fig.add_scatter(x=cond['first_plan'], y=cond['rungs'], name='rungs above',
                line=dict(color=PALETTE[3], width=3), secondary_y=True)
fig.update_yaxes(title='upgrade rate', tickformat='.0%')
fig.update_yaxes(title='rungs available above', secondary_y=True)
fig.update_layout(title='is the upgrade rate about the customer, or about the ladder?')
fig.show()

upgrade rate, all customers:        14.7%
upgrade rate, only those who can:   15.1%

upgrade rate among customers with headroom:
 plan_rank          first_plan    n  rungs  upgrade_rate
      1.00    Higgsfield Basic 9419   3.00          0.04
      2.00      Higgsfield Pro 6825   2.00          0.21
      3.00 Higgsfield Ultimate 2081   1.00          0.49

rungs vs upgrade rate (3 rungs - too few for a meaningful correlation):
  Higgsfield Basic       3 rungs above -> 3.7%
  Higgsfield Pro         2 rungs above -> 20.6%
  Higgsfield Ultimate    1 rungs above -> 49.0%
-> monotone in headroom: likely the LADDER talking, not the customer


In [11]:
# where on the ladder do customers move? one rung or several?
# NOTE: the "from" plan must be the plan held IMMEDIATELY BEFORE the upgrade, not
# base.first_plan. a customer who upgrades twice has a different starting rung the
# second time, and using first_plan produces phantom same-plan "upgrades".
# ch carries plan_dir but not prev_plan (it was built on a subscription-only slice),
# so rebuild the preceding plan here, over subscription charges only.
subs = ch[~ch['is_credits']].sort_values(['customer', 'revenue_time']).copy()
subs['prev_plan'] = subs.groupby('customer')['subscription_plan'].shift()
ch_prev = subs.set_index('charge_id')['prev_plan']

moves = ch[ch['plan_dir'] > 0].copy()
moves['prev_plan'] = moves['charge_id'].map(ch_prev)
moves = moves.dropna(subset=['prev_plan'])
moves['from_rank'] = moves['prev_plan'].map(RANK)
moves['to_rank'] = moves['subscription_plan'].map(RANK)
moves = moves.dropna(subset=['from_rank', 'to_rank'])
moves['step'] = moves['to_rank'] - moves['from_rank']

assert (moves['step'] > 0).all(), 'an upgrade with step <= 0 means prev_plan is wrong'
print(f'upgrade steps taken:\n{moves["step"].value_counts().sort_index()}')
print(f'one-rung moves: {(moves["step"] == 1).mean():.1%}')

flow = moves.groupby(['prev_plan', 'subscription_plan']).size().rename('n').reset_index()
print(f'\nupgrade paths:\n{flow.sort_values("n", ascending=False).to_string(index=False)}')

mat = moves.pivot_table(index='prev_plan', columns='subscription_plan',
                        values='charge_id', aggfunc='size')
mat = mat.reindex(index=[p for p in LADDER if p in mat.index],
                  columns=[p for p in LADDER if p in mat.columns])
fig = px.imshow(mat.fillna(0), text_auto=True, aspect='auto',
                color_continuous_scale='Purples', title='upgrade flows: from plan -> to plan')
fig.show()

mat_pct = mat.fillna(0) / mat.fillna(0).sum().sum()
fig = px.imshow(mat_pct, text_auto='.0%', aspect='auto',
                color_continuous_scale='Purples',
                title='upgrade flows: share of all upgrades')
fig.show()

print(f'\nrevenue per upgrade by step size:\n{moves.groupby("step")["sales_amount"].agg(["size","mean","sum"]).round(2)}')

# which rung is hardest to leave? conversion out of each plan, given headroom exists
left = moves.groupby('prev_plan')['customer'].nunique().rename('upgraded_out')
held = xfeat[xfeat['can_upgrade'] == 1].groupby('first_plan').size().rename('customers')
rungs = pd.concat([held, left], axis=1).dropna()
rungs['rate'] = rungs['upgraded_out'] / rungs['customers']
rungs = rungs.reindex([p for p in LADDER if p in rungs.index])
print(f'\nupgrade-out rate by rung:\n{rungs.round(3)}')

upgrade steps taken:
step
1    3110
2     194
3       2
Name: count, dtype: int64
one-rung moves: 94.1%

upgrade paths:
          prev_plan   subscription_plan    n
   Higgsfield Basic      Higgsfield Pro 1854
     Higgsfield Pro Higgsfield Ultimate 1198
   Higgsfield Basic Higgsfield Ultimate  180
Higgsfield Ultimate  Higgsfield Creator   58
     Higgsfield Pro  Higgsfield Creator   14
   Higgsfield Basic  Higgsfield Creator    2



revenue per upgrade by step size:
      size   mean        sum
step                        
1     3110  57.26 178,063.87
2      194 167.58  32,511.37
3        2 100.32     200.64

upgrade-out rate by rung:
                     customers  upgraded_out  rate
Higgsfield Basic          9419          1986  0.21
Higgsfield Pro            6825          1174  0.17
Higgsfield Ultimate       2081            57  0.03


In [12]:
# for each plan: who started there vs who expanded in from a lower rung?
first_sub = (subs.sort_values(['customer', 'revenue_time'])
             .groupby('customer').first()['subscription_plan']
             .rename('first_plan'))
started = first_sub.value_counts().rename('started_on')
# unique customers who upgraded INTO each plan (destination of an upgrade)
upgraded_into = moves.groupby('subscription_plan')['customer'].nunique().rename('upgraded_into')

origin = pd.concat([started, upgraded_into], axis=1).fillna(0).astype(int)
origin = origin.reindex([p for p in LADDER if p in origin.index])
origin['arrivals'] = origin['started_on'] + origin['upgraded_into']
origin['share_started'] = origin['started_on'] / origin['arrivals']
origin['share_expanded_in'] = origin['upgraded_into'] / origin['arrivals']
origin['expanded_per_starter'] = origin['upgraded_into'] / origin['started_on'].replace(0, np.nan)
print('plan mix: original starters vs expanded-in (customer counts; disjoint per plan)')
print(origin.round(3).to_string())

long = origin.reset_index(names='plan')[['plan', 'started_on', 'upgraded_into']].melt(
    id_vars='plan', var_name='origin', value_name='customers')
long['origin'] = long['origin'].map({'started_on': 'started on plan', 'upgraded_into': 'expanded into plan'})
fig = px.bar(long, x='plan', y='customers', color='origin', barmode='stack',
             title='who is on each plan: started there vs expanded in')
fig.show()

fig = px.bar(origin.reset_index(names='plan'), x='plan', y='share_expanded_in',
             title='share of plan arrivals that came via upgrade (not original signup)')
fig.update_yaxes(tickformat='.0%', range=[0, 1])
fig.show()

# --- credits usage ---
cred = ch[ch['is_credits']].copy()
print(f'\ncredits charges: {len(cred):,} | buyers: {cred["customer"].nunique():,} | '
      f'revenue: ${cred["sales_amount"].sum():,.0f} '
      f'({cred["sales_amount"].sum() / ch["sales_amount"].sum():.1%} of total)')

# plan held at purchase = last subscription charge on/before the credits charge
# (avoid merge_asof+by: pandas requires a globally sorted time key)
cred = cred.sort_values(['customer', 'revenue_time']).copy()
plan_at = []
for cust, g in cred.groupby('customer', sort=False):
    hist = subs.loc[subs['customer'] == cust, ['revenue_time', 'subscription_plan']]
    if hist.empty:
        plan_at.extend([first_sub.get(cust)] * len(g))
        continue
    times = hist['revenue_time'].to_numpy()
    plans = hist['subscription_plan'].to_numpy()
    idx = np.searchsorted(times, g['revenue_time'].to_numpy(), side='right') - 1
    vals = [plans[i] if i >= 0 else first_sub.get(cust) for i in idx]
    plan_at.extend(vals)
cred['plan_at_buy'] = plan_at
cred['plan_at_buy'] = cred['plan_at_buy'].fillna(cred['customer'].map(first_sub))

by_plan = cred.groupby('plan_at_buy').agg(
    charges=('charge_id', 'size'), buyers=('customer', 'nunique'),
    revenue=('sales_amount', 'sum'), arpu=('sales_amount', 'mean')).reindex(LADDER).dropna(how='all')
print(f'\ncredits by plan held at purchase:\n{by_plan.round(2).to_string()}')

n_per = cred.groupby('customer').size()
print(f'\npurchases per buyer:\n{n_per.describe(percentiles=[.5, .9, .99]).round(2)}')

# plots
cred_m = (cred.set_index('revenue_time')
          .groupby([pd.Grouper(freq='MS'), 'plan_at_buy'])['sales_amount']
          .sum().rename('revenue').reset_index())
fig = px.bar(cred_m, x='revenue_time', y='revenue', color='plan_at_buy',
             title='monthly credits revenue by plan held at purchase',
             category_orders={'plan_at_buy': LADDER})
fig.show()

fig = px.bar(by_plan.reset_index(), x='plan_at_buy', y='buyers',
             title='unique credits buyers by plan held at purchase')
fig.show()

hist = n_per.clip(upper=10).value_counts().sort_index().reset_index()
hist.columns = ['n_purchases', 'buyers']
fig = px.bar(hist, x='n_purchases', y='buyers',
             title='credits purchase frequency (10+ clipped)')
fig.show()

# intensity: credit buyers vs not (uses xfeat if already built)
if 'xfeat' in dir() and 'jobs_28' in getattr(xfeat, 'columns', []):
    tmp = xfeat.copy()
    tmp['n_credits'] = tmp.index.map(n_per).fillna(0)
    tmp['credit_buyer'] = (tmp['n_credits'] > 0).astype(int)
    aggs = {'n': ('jobs_28', 'size'), 'jobs_28': ('jobs_28', 'median')}
    if 'days_28' in tmp.columns:
        aggs['days_28'] = ('days_28', 'median')
    if 'renewed' in tmp.columns:
        aggs['renewed'] = ('renewed', 'mean')
    if 'upgraded' in tmp.columns:
        aggs['upgraded'] = ('upgraded', 'mean')
    cmp = tmp.groupby('credit_buyer').agg(**aggs)
    print(f'\ncredit buyers vs not (xfeat):\n{cmp.round(3)}')
    fig = px.box(tmp[tmp['jobs_28'] >= 0].sample(min(8000, len(tmp)), random_state=1),
                 x='credit_buyer', y='jobs_28',
                 title='jobs in first 28d: credit buyers vs not')
    fig.show()


plan mix: original starters vs expanded-in (customer counts; disjoint per plan)
                     started_on  upgraded_into  arrivals  share_started  share_expanded_in  expanded_per_starter
Higgsfield Basic          11420              0     11420           1.00               0.00                  0.00
Higgsfield Pro             6803           1813      8616           0.79               0.21                  0.27
Higgsfield Ultimate        1270           1326      2596           0.49               0.51                  1.04
Higgsfield Creator           37             73       110           0.34               0.66                  1.97



credits charges: 6,773 | buyers: 2,487 | revenue: $194,794 (9.1% of total)



credits by plan held at purchase:
                     charges  buyers   revenue  arpu
plan_at_buy                                         
Higgsfield Basic        2160    1063 31,380.46 14.53
Higgsfield Pro          2952    1081 78,945.00 26.74
Higgsfield Ultimate     1608     544 80,823.87 50.26
Higgsfield Creator        47      14  3,537.81 75.27

purchases per buyer:
count   2,487.00
mean        2.72
std         4.10
min         1.00
50%         1.00
90%         6.00
99%        20.00
max        69.00
dtype: float64


### Insight — the plan ladder is ordinal, and that changes the upgrade read

**Basic < Pro < Ultimate < Creator.** Treating `first_plan` as a flat category hides the mechanical part of the upgrade rate: Basic has three rungs above it, Creator has none. Creator customers **cannot upgrade** → including them in an upgrade denominator measures the ladder, not the customer.

Three corrections that follow:

- **Upgrade rate must be conditional on headroom.** Compare only customers with somewhere to go. If the conditional rate is monotone in `rungs_above`, the ranking is the ladder talking.
- **`rungs_above` replaces `first_plan` in the route model.** It's the causally relevant part of plan, and it's one ordinal number instead of four dummies. The three-way AUC comparison says whether plan carries anything beyond position — price point, feature set — or was only ever a proxy for it.
- **Upgrades are near-always one rung.** → the ladder is walked, not jumped → the lever is the price gap between *adjacent* tiers, not the top-tier price. Which rung has the lowest upgrade-out rate is the tier whose next step is priced wrong.

**→ for Creator customers, credits are the only expansion route that exists.** Any expansion programme aimed at them is a credits programme by definition.

## Generation frame (shared by CJM and §5)

Jobs in `[-1, 28]` days after first charge. Built once here, reused by the journey funnel and the leak-free feature table.

In [13]:
# extra imports for this section
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

LADDER = ['Higgsfield Basic', 'Higgsfield Pro', 'Higgsfield Ultimate', 'Higgsfield Creator']
RANK = {p: i + 1 for i, p in enumerate(LADDER)}

cust_lu = customers_df.dropna(subset=['customer']).drop_duplicates('user_id')
gen = generations_df.merge(
    cust_lu[['user_id', 'customer', 'created_at', 'platform', 'region']],
    on='user_id', how='inner')
gen = gen.merge(
    base[['customer', 'first_pay', 'first_plan', 'first_period', 'tenure_days']],
    on='customer', how='inner')
gen['days_since_pay'] = (gen['in_progress_at'] - gen['first_pay']).dt.total_seconds() / 86400
gen['hour'] = gen['in_progress_at'].dt.hour  # UTC, not local — proxy only
gen['dow'] = gen['in_progress_at'].dt.dayofweek  # 0=Mon, UTC weekday of the job
gen['is_weekday'] = (gen['dow'] < 5).astype(int)
gen['is_workhour'] = gen['hour'].between(9, 17).astype(int)

# first-renewal / month-2 still-paying (subscription charges only)
sub_pay = (charges_df[~charges_df['is_credits']][['customer', 'revenue_time']]
           .merge(base[['customer', 'first_pay']], on='customer'))
sub_pay['days'] = (sub_pay['revenue_time'] - sub_pay['first_pay']).dt.total_seconds() / 86400
renew_30 = set(sub_pay[(sub_pay['days'] > 1) & (sub_pay['days'] <= 30 + GRACE)]['customer'])
renew_60 = set(sub_pay[(sub_pay['days'] > 30) & (sub_pay['days'] <= 60 + GRACE)]['customer'])

exp_ev = ch.loc[ch['is_credits'] | (ch['plan_dir'] > 0),
                ['customer', 'revenue_time', 'is_credits', 'sales_amount']].merge(
    base[['customer', 'first_pay']], on='customer')
exp_ev['days_in'] = (exp_ev['revenue_time'] - exp_ev['first_pay']).dt.total_seconds() / 86400
exp_ev['kind'] = np.where(exp_ev['is_credits'], 'credits', 'upgrade')

print(f'gen rows: {len(gen):,} | renewed@30 {len(renew_30):,} | renewed@60 {len(renew_60):,} | exp events {len(exp_ev):,}')


gen rows: 4,948,652 | renewed@30 9,133 | renewed@60 9,561 | exp events 10,079


In [14]:
# CJM needs a customer-level frame with 28d usage and first-renewal, aligned to gen
model_df = base[(base['first_period'] == 'month') & (base['tenure_days'] >= 30 + GRACE)].copy()
model_df = model_df.set_index('customer')
g28_cjm = gen[(gen['days_since_pay'] >= -1) & (gen['days_since_pay'] <= 28)]
model_df['jobs_28'] = g28_cjm.groupby('customer').size()
model_df['jobs_28'] = model_df['jobs_28'].fillna(0)
model_df['video_share_28'] = g28_cjm.groupby('customer')['is_video'].mean()
model_df['renewed'] = model_df.index.isin(renew_30).astype(int)

exp_df = pd.DataFrame(index=model_df.index)
exp_df['expanded'] = model_df.index.isin(set(exp_ev['customer'])).astype(int)
print(f'CJM frame: {len(model_df):,} monthly observable | generated {(model_df["jobs_28"]>0).mean():.1%} | '
      f'renewed {model_df["renewed"].mean():.1%} | expanded {exp_df["expanded"].mean():.1%}')


CJM frame: 16,091 monthly observable | generated 99.0% | renewed 50.9% | expanded 22.6%


## CJM

In [15]:
# a funnel is only honest if each stage is a SUBSET of the previous one.
# stages here are cumulative gates, not independent flags - otherwise the chart
# can go up between steps and stops meaning anything.
j = model_df.copy()
j['exp_flag'] = exp_df['expanded'].reindex(j.index).fillna(0).astype(int)

sess = gen[gen['customer'].isin(j.index)].copy()
sess['days_since_pay'] = (sess['in_progress_at'] - sess['first_pay']).dt.total_seconds() / 86400
sess = sess[(sess['days_since_pay'] >= -1) & (sess['days_since_pay'] <= 28)]
day_count = sess.groupby('customer')['in_progress_at'].apply(lambda s: s.dt.date.nunique())
j['days_28d'] = day_count.reindex(j.index).fillna(0)

g1 = pd.Series(True, index=j.index)                  # paid
g2 = g1 & (j['jobs_28'] > 0)                         # generated
g3 = g2 & (j['days_28d'] >= 2)                       # came back
g4 = g3 & (j['days_28d'] >= 5)                       # habit
g5 = g4 & (j['renewed'] == 1)                        # renewed

stages = pd.DataFrame({
    'stage': ['1. paid', '2. generated', '3. came back (2+ days)',
              '4. habit (5+ days)', '5. renewed'],
    'customers': [int(g.sum()) for g in [g1, g2, g3, g4, g5]]})
stages['share_of_paid'] = stages['customers'] / len(j)
stages['step_conv'] = stages['customers'] / stages['customers'].shift()
stages['lost'] = stages['customers'].shift() - stages['customers']
print(stages.round(3).to_string(index=False))

assert stages['customers'].is_monotonic_decreasing, 'stages are not nested'

fig = px.funnel(stages, x='customers', y='stage', title='customer journey funnel (nested gates)')
fig.show()

worst = stages.loc[stages['lost'].idxmax()]
print(f'\nbiggest drop: {worst["stage"]} loses {int(worst["lost"]):,} '
      f'({1 - worst["step_conv"]:.1%} of the previous stage)')

# expansion is NOT a funnel stage - it is a branch off habit, so it is counted separately
print(f'\nexpansion (a branch, not a gate): {int(j["exp_flag"].sum()):,} customers '
      f'({j["exp_flag"].mean():.1%})')
print(f'  of those with habit: {j.loc[g4, "exp_flag"].mean():.1%}')
print(f'  of those without:    {j.loc[~g4, "exp_flag"].mean():.1%}')

                 stage  customers  share_of_paid  step_conv     lost
               1. paid      16091           1.00        NaN      NaN
          2. generated      15923           0.99       0.99   168.00
3. came back (2+ days)      11120           0.69       0.70 4,803.00
    4. habit (5+ days)       4660           0.29       0.42 6,460.00
            5. renewed       2845           0.18       0.61 1,815.00



biggest drop: 4. habit (5+ days) loses 6,460 (58.1% of the previous stage)

expansion (a branch, not a gate): 3,644 customers (22.6%)
  of those with habit: 47.7%
  of those without:    12.4%


In [16]:
# conditional renewal - how much does reaching each stage change the outcome?
checks = {
    'generated at all': j['jobs_28'] > 0,
    'came back (2+ days)': j['days_28d'] >= 2,
    'habit (5+ days)': j['days_28d'] >= 5,
    'made a video': j['video_share_28'].fillna(0) > 0,
    'expanded': j['exp_flag'] == 1,
}
rows = []
for name, m in checks.items():
    if m.sum() < 20 or (~m).sum() < 20:
        continue
    rows.append({'stage': name, 'n_reached': int(m.sum()),
                 'renewal_if_reached': j.loc[m, 'renewed'].mean(),
                 'renewal_if_not': j.loc[~m, 'renewed'].mean(),
                 'lift': j.loc[m, 'renewed'].mean() - j.loc[~m, 'renewed'].mean()})
cond = pd.DataFrame(rows).sort_values('lift', ascending=False)
print(cond.round(3).to_string(index=False))

fig = go.Figure()
fig.add_bar(x=cond['stage'], y=cond['renewal_if_reached'], name='reached', marker_color=PALETTE[1])
fig.add_bar(x=cond['stage'], y=cond['renewal_if_not'], name='did not reach', marker_color=PALETTE[6])
fig.update_layout(barmode='group', title='renewal by journey stage reached', yaxis_tickformat='.0%')
fig.show()

              stage  n_reached  renewal_if_reached  renewal_if_not  lift
           expanded       3644                0.68            0.46  0.22
    habit (5+ days)       4660                0.61            0.47  0.14
   generated at all      15923                0.51            0.38  0.14
came back (2+ days)      11120                0.53            0.46  0.07
       made a video      14965                0.51            0.49  0.02


In [17]:
# stage 6: the fork. of customers who showed limit pressure, who paid more and who left?
pressure = j[j['jobs_28'] >= j['jobs_28'].quantile(0.75)]
print(f'high-usage customers (top quartile of jobs in 28d): {len(pressure):,}')

fork = pd.DataFrame({
    'outcome': ['expanded', 'renewed without expanding', 'churned'],
    'customers': [
        int(pressure['exp_flag'].sum()),
        int(((pressure['exp_flag'] == 0) & (pressure['renewed'] == 1)).sum()),
        int((pressure['renewed'] == 0).sum()),
    ]})
fork['share'] = fork['customers'] / len(pressure)
print(f'\n{fork.round(3).to_string(index=False)}')

fig = px.pie(fork, values='customers', names='outcome', hole=0.45,
             title='the fork: what high-usage customers did')
fig.show()

base_rates = pd.DataFrame({
    'group': ['high usage', 'everyone else'],
    'expansion': [pressure['exp_flag'].mean(), j.loc[~j.index.isin(pressure.index), 'exp_flag'].mean()],
    'churn': [1 - pressure['renewed'].mean(), 1 - j.loc[~j.index.isin(pressure.index), 'renewed'].mean()],
})
print(f'\n{base_rates.round(3).to_string(index=False)}')
print(f'\n-> high-usage customers churn at {base_rates.loc[0,"churn"]:.1%} vs '
      f'{base_rates.loc[1,"churn"]:.1%} for everyone else')

high-usage customers (top quartile of jobs in 28d): 4,034

                  outcome  customers  share
                 expanded       1901   0.47
renewed without expanding       1000   0.25
                  churned       1605   0.40



        group  expansion  churn
   high usage       0.47   0.40
everyone else       0.14   0.52

-> high-usage customers churn at 39.8% vs 52.2% for everyone else


---
# 5. Hypothesis tests → behavioural segments

Same data rules as §2–3: clock starts at **first charge**; retention target is **first renewal (~30d)** among customers with enough tenure; expansion is **upgrade** vs **credits**, not one blob.

Two methods:

1. **Bivariate tests** — each H1 vs its outcome (Mann–Whitney for continuous, χ² for categorical).
2. **ML** — CatBoost + SHAP, and a **shallow** random forest (`max_depth=4`) with permutation importance and 95% CI.

Features for the 30-day outcomes are built only on generations in the **first 28 days**. Hypotheses that need month 2 use a later window and a later outcome (no leakage).

Do not pool targets. Retention: **M1 renewal** vs **post-M1** (among those who already renewed). Expansion: **upgrade** vs **credits** (and **both**). Same feature set, separate tests and models — a shape change means the same early behaviour can map to different later outcomes.


## 5.0 Frame and features


In [18]:
# short prompt cut from the actual length distribution (chars, not words)
# same window as features: jobs in [-1, 28] after first charge
g28 = gen[(gen['days_since_pay'] >= -1) & (gen['days_since_pay'] <= 28)].copy()
pl = g28['prompt_length'].dropna()
pl_pos = pl[pl > 0]

pct = [0.01, 0.05, 0.10, 0.25, 0.50, 0.75, 0.90, 0.95, 0.99]
print('prompt_length, 28d jobs:')
print(pl.describe(percentiles=pct).round(1))
print(f'empty (0): {(pl == 0).mean():.1%}  n={len(pl):,}')
print('\nnon-empty only:')
print(pl_pos.describe(percentiles=pct).round(1))

# left tail of non-empty prompts = low-effort / stub. p10, not p25 (that is just "below typical")
SHORT_Q = 0.10
SHORT_PROMPT = float(pl_pos.quantile(SHORT_Q))
share = (pl <= SHORT_PROMPT).mean()
print(f'\nSHORT_PROMPT = p{int(SHORT_Q*100)} of non-empty = {SHORT_PROMPT:.0f} chars  '
      f'({share:.1%} of all 28d jobs, including empty)')

clip = pl.clip(upper=pl.quantile(0.99))
# sample for the plot — the full 28d job set is millions of rows and bloats the ipynb
clip_plot = clip.sample(n=min(80_000, len(clip)), random_state=1) if len(clip) > 80_000 else clip
fig = px.histogram(clip_plot, nbins=80,
                   title='prompt_length, 28d jobs (clipped at p99; sample for plot)')
fig.add_vline(x=SHORT_PROMPT, line_dash='dash',
              annotation_text=f'p{int(SHORT_Q*100)} = {SHORT_PROMPT:.0f}')
fig.show()


prompt_length, 28d jobs:
count   2,370,392.00
mean          764.70
std           494.40
min             0.00
1%              0.00
5%             25.00
10%            63.00
25%           428.00
50%           766.00
75%         1,063.00
90%         1,392.00
95%         1,554.00
99%         1,951.00
max        47,015.00
Name: prompt_length, dtype: float64
empty (0): 3.0%  n=2,370,392

non-empty only:
count   2,298,811.00
mean          788.50
std           483.00
min             1.00
1%             16.00
5%             45.00
10%            91.00
25%           501.00
50%           780.00
75%         1,078.00
90%         1,400.00
95%         1,560.00
99%         1,962.00
max        47,015.00
Name: prompt_length, dtype: float64



SHORT_PROMPT = p10 of non-empty = 91 chars  (12.8% of all 28d jobs, including empty)


In [19]:
# leak-free features from generations in [-1, 28] days after first charge
g28 = gen[(gen['days_since_pay'] >= -1) & (gen['days_since_pay'] <= 28)].copy()
if 'dow' not in g28.columns:
    g28['dow'] = g28['in_progress_at'].dt.dayofweek
# SHORT_PROMPT from the distribution cell above (p10 of non-empty prompt_length)
if 'SHORT_PROMPT' not in dir():
    SHORT_PROMPT = float(g28.loc[g28['prompt_length'] > 0, 'prompt_length'].quantile(0.10))
print(f'short-prompt cut: {SHORT_PROMPT:.0f} chars')

def agg_window(df, lo, hi, pfx):
    w = df[(df['days_since_pay'] >= lo) & (df['days_since_pay'] <= hi)]
    a = w.groupby('customer').agg(
        jobs=('job_id', 'size'),
        days=('in_progress_at', lambda s: s.dt.date.nunique()),
        prompt_med=('prompt_length', 'median'),
        prompt_mean=('prompt_length', 'mean'),
        prompt_max=('prompt_length', 'max'),
        short=('prompt_length', lambda s: (s <= SHORT_PROMPT).mean()),
        weekday=('is_weekday', 'mean'),
        weekend=('is_weekday', lambda s: 1 - s.mean()),
        workhour=('is_workhour', 'mean'),
        hour_std=('hour', 'std'),
        video=('is_video', 'mean'),
        n_video=('is_video', 'sum'),
        fail=('failed', 'mean'),
        dow_med=('dow', 'median'),
    )
    a.columns = [f'{pfx}_{c}' for c in a.columns]
    return a

F = pd.DataFrame(index=pd.Index(base['customer'].unique(), name='customer'))
F = F.join(agg_window(g28, -1, 7, 'w1'))
F = F.join(agg_window(g28, -1, 14, 'd14'))
F = F.join(agg_window(g28, 14, 21, 'w3'))
F = F.join(agg_window(g28, 21, 28, 'w4'))
F = F.join(agg_window(g28, -1, 15, 'd15'))
F = F.join(agg_window(g28, -1, 28, 'm1'))

# video/image by window: share and ratio (image = jobs - video)
for pfx in ['w1', 'd14', 'w3', 'w4', 'm1']:
    if f'{pfx}_n_video' in F.columns and f'{pfx}_jobs' in F.columns:
        img = (F[f'{pfx}_jobs'] - F[f'{pfx}_n_video']).clip(lower=0)
        F[f'{pfx}_video_image'] = F[f'{pfx}_n_video'] / img.replace(0, np.nan)
F['video_image_wow'] = F['w3_video_image'] - F['w1_video_image']

pay = base.set_index('customer')['first_pay']
F['first_pay'] = pay
first_gen = g28.groupby('customer')['in_progress_at'].min()
last_gen = g28.groupby('customer')['in_progress_at'].max()
F['hours_to_first_gen'] = (first_gen - pay).dt.total_seconds() / 3600
F['days_to_first_gen'] = F['hours_to_first_gen'] / 24
F['never_generated'] = F['hours_to_first_gen'].isna().astype(int)
F['recency_at_28'] = (pay + pd.Timedelta(days=28) - last_gen).dt.total_seconds() / 86400

# binge vs habit: (1) share of W1 jobs on the heaviest day  (2) variance of jobs per active day
day_jobs = (g28.assign(d=g28['in_progress_at'].dt.date)
            .groupby(['customer', 'd']).size().rename('n'))
peak_w1 = (g28[g28['days_since_pay'] <= 7]
           .assign(d=lambda x: x['in_progress_at'].dt.date)
           .groupby(['customer', 'd']).size().groupby('customer').max())
F['binge_w1'] = (peak_w1 / F['w1_jobs']).clip(upper=1)
F['jobs_per_day_var'] = day_jobs.groupby('customer').var()
F['jobs_per_day_mean'] = day_jobs.groupby('customer').mean()

# work vs play: weekday / weekend job counts (not just share)
wd = g28.groupby(['customer', 'is_weekday']).size().unstack(fill_value=0)
if 1 not in wd.columns: wd[1] = 0
if 0 not in wd.columns: wd[0] = 0
F['weekday_jobs'] = wd[1]
F['weekend_jobs'] = wd[0]
F['weekday_weekend_ratio'] = F['weekday_jobs'] / F['weekend_jobs'].clip(lower=1)

# max silent gap + cool-off: zero-generation days in the last 10d of month 1 (days 19-28)
dates = (g28.assign(d=g28['in_progress_at'].dt.normalize())
         [['customer', 'd']].drop_duplicates().sort_values(['customer', 'd']))
dates['gap'] = dates.groupby('customer')['d'].diff().dt.days
F['max_gap_28'] = dates.groupby('customer')['gap'].max()
gap_start = F['hours_to_first_gen'] / 24
F['max_gap_28'] = pd.concat([F['max_gap_28'], gap_start, F['recency_at_28']], axis=1).max(axis=1)

active_last10 = (g28[g28['days_since_pay'].between(19, 28)]
                 .groupby('customer')['in_progress_at'].apply(lambda s: s.dt.date.nunique()))
F['cooloff_last10'] = 10 - active_last10.reindex(F.index).fillna(0)

# momentum: WoW = (W3 - W1) / W1 ; prompt evolution = W3 mean - W1 mean (as specified)
F['wow_growth'] = (F['w3_jobs'] - F['w1_jobs']) / F['w1_jobs'].replace(0, np.nan)
F['prompt_evolution'] = F['w3_prompt_mean'] - F['w1_prompt_mean']

# end-of-cycle spike: last 3 days vs daily mean
g_last3 = g28[g28['days_since_pay'] >= 27]
F['endcycle_jobs'] = g_last3.groupby('customer').size().reindex(F.index).fillna(0)
F['endcycle_spike'] = F['endcycle_jobs'] / (F['m1_jobs'].replace(0, np.nan) / 28)

# context from base
ctx_cols = [c for c in ['platform', 'region', 'first_plan', 'first_period',
                        'tenure_days', 'created_at', 'revenue', 'first_pay']
            if c in base.columns]
ctx = base.set_index('customer')[ctx_cols].copy()
F = F.join(ctx, rsuffix='_ctx')
if 'first_pay' not in F.columns and 'first_pay_ctx' in F.columns:
    F['first_pay'] = F['first_pay_ctx']

# flags first — these must exist even if later lines fail
F['annual'] = (F['first_period'].astype(str) == 'year').astype(int)
F['plan_rank'] = F['first_plan'].map(RANK)
F['rungs_above'] = 4 - F['plan_rank']
F['can_upgrade'] = (F['rungs_above'].fillna(0) > 0).astype(int)
F['desktop'] = F['platform'].astype(str).str.lower().eq('desktop').astype(int)

F['signup_dow'] = F['created_at'].dt.dayofweek  # 0=Mon
F['signup_month'] = F['created_at'].dt.month
F['signup_weekend'] = (F['signup_dow'] >= 5).astype(int)
if 'first_pay' in F.columns and 'created_at' in F.columns:
    F['hours_signup_to_pay'] = (F['first_pay'] - F['created_at']).dt.total_seconds() / 3600
    F['days_signup_to_pay'] = F['hours_signup_to_pay'] / 24

# quota proxy: no credit cap in the data. burn = jobs by day 15 vs own plan's p75 of 28d jobs
plan_p75 = F.groupby('first_plan')['m1_jobs'].transform(lambda s: s.quantile(0.75)).replace(0, np.nan)
F['quota_burn_d15'] = F['d15_jobs'] / plan_p75
F['frontload_d15'] = F['d15_jobs'] / F['m1_jobs'].replace(0, np.nan)

# fill activity zeros for people who never generated
for c in ['w1_jobs', 'w1_days', 'd14_jobs', 'd14_days', 'm1_jobs', 'm1_days',
          'w3_jobs', 'w4_jobs', 'd15_jobs', 'binge_w1', 'endcycle_jobs',
          'weekday_jobs', 'weekend_jobs', 'cooloff_last10']:
    if c in F.columns:
        F[c] = F[c].fillna(0)

print('can_upgrade in F:', 'can_upgrade' in F.columns, F['can_upgrade'].mean() if 'can_upgrade' in F.columns else None)
print(f'feature frame: {F.shape} | generated in 28d: {(F["never_generated"]==0).mean():.1%}')
print(F[['hours_to_first_gen', 'w1_days', 'd14_days', 'm1_days', 'binge_w1',
         'jobs_per_day_var', 'weekday_weekend_ratio', 'cooloff_last10',
         'quota_burn_d15', 'days_signup_to_pay',
         'w1_video_image', 'd14_video_image', 'm1_video_image',
         'w1_dow_med', 'd14_dow_med', 'm1_dow_med']].describe().round(2))


short-prompt cut: 91 chars


can_upgrade in F: True 0.9382039729674381
feature frame: (19532, 127) | generated in 28d: 98.7%
       hours_to_first_gen   w1_days  d14_days   m1_days  binge_w1  jobs_per_day_var  weekday_weekend_ratio  cooloff_last10  quota_burn_d15  days_signup_to_pay  w1_video_image  d14_video_image  m1_video_image  \
count           19,271.00 19,532.00 19,532.00 19,532.00 19,532.00         13,642.00              19,271.00       19,532.00       19,215.00           19,532.00        8,197.00         8,746.00        9,352.00   
mean                 4.93      2.32      3.11      4.36      0.74          2,492.10                  20.15            9.21            1.12                7.73            2.59             3.16            3.99   
std                 34.29      1.62      2.69      4.67      0.26         62,455.99                  93.37            1.70            3.34               20.39            8.65            11.39           20.97   
min                -24.00      0.00      0.00      0.00     

In [20]:
# outcomes (aligned to F index)
# rebuild flags here so tests don't depend on a half-finished feature cell
if 'platform' in F.columns and 'desktop' not in F.columns:
    F['desktop'] = F['platform'].astype(str).str.lower().eq('desktop').astype(int)
if 'first_period' in F.columns and 'annual' not in F.columns:
    F['annual'] = (F['first_period'].astype(str) == 'year').astype(int)
if 'first_plan' in F.columns and 'can_upgrade' not in F.columns:
    F['plan_rank'] = F['first_plan'].map(RANK)
    F['rungs_above'] = 4 - F['plan_rank']
    F['can_upgrade'] = (F['rungs_above'].fillna(0) > 0).astype(int)

F['renewed_30'] = F.index.isin(renew_30).astype(int)
F['renewed_60'] = F.index.isin(renew_60).astype(int)

first_exp = (exp_ev.sort_values('days_in').groupby('customer').first()
             [['days_in', 'kind']].rename(columns={'days_in': 'first_exp_days', 'kind': 'first_exp_kind'}))
n_exp = exp_ev.groupby('customer').size().rename('n_exp')
n_up = exp_ev[exp_ev['kind']=='upgrade'].groupby('customer').size().rename('n_upgrades')
n_cr = exp_ev[exp_ev['kind']=='credits'].groupby('customer').size().rename('n_credits')
exp_rev = exp_ev.groupby('customer')['sales_amount'].sum().rename('exp_revenue')

F = F.join(first_exp).join(n_exp).join(n_up).join(n_cr).join(exp_rev)
F['n_exp'] = F['n_exp'].fillna(0)
F['n_upgrades'] = F['n_upgrades'].fillna(0)
F['n_credits'] = F['n_credits'].fillna(0)
F['exp_revenue'] = F['exp_revenue'].fillna(0)
F['expanded_30'] = ((F['first_exp_days'] >= 0) & (F['first_exp_days'] <= 30)).fillna(False).astype(int)
F['upgraded_30'] = F.index.isin(set(exp_ev.loc[exp_ev['kind'].eq('upgrade') & exp_ev['days_in'].between(0, 30), 'customer'])).astype(int)
F['credits_30'] = F.index.isin(set(exp_ev.loc[exp_ev['kind'].eq('credits') & exp_ev['days_in'].between(0, 30), 'customer'])).astype(int)
F['both_30'] = ((F['upgraded_30']==1) & (F['credits_30']==1)).astype(int)
F['credits_after_60'] = F.index.isin(set(exp_ev.loc[exp_ev['kind'].eq('credits') & (exp_ev['days_in'] > 60), 'customer'])).astype(int)
F['expanded_after_30'] = F.index.isin(set(exp_ev.loc[exp_ev['days_in'] > 30, 'customer'])).astype(int)

# month-2 usage (only for hypotheses that need it; outcome after day 60)
g_m2 = gen[(gen['days_since_pay'] > 30) & (gen['days_since_pay'] <= 60)]
F['jobs_m2'] = g_m2.groupby('customer').size()
F['jobs_m2'] = F['jobs_m2'].fillna(0)
F['mom_growth'] = (F['jobs_m2'] - F['m1_jobs']) / F['m1_jobs'].replace(0, np.nan)

# eligibility flags
F['obs_30'] = (F['tenure_days'] >= 30 + GRACE).astype(int)
F['obs_60'] = (F['tenure_days'] >= 60 + GRACE).astype(int)
F['obs_90'] = (F['tenure_days'] >= 90 + GRACE).astype(int)
F['monthly'] = (F['first_period'] == 'month').astype(int)

print('base rates among monthly + observable 30d:')
m = F[(F['obs_30']==1) & (F['monthly']==1)]
print(f'  n={len(m):,}  renewal={m["renewed_30"].mean():.1%}  expand30={m["expanded_30"].mean():.1%}  '
      f'upgrade30={m["upgraded_30"].mean():.1%}  credits30={m["credits_30"].mean():.1%}  both30={m["both_30"].mean():.1%}')
post = F[(F['obs_60']==1) & (F['monthly']==1) & (F['renewed_30']==1)]
print(f'post-M1 sample (renewed@30, tenure≥60d): n={len(post):,}  renewed_60={post["renewed_60"].mean():.1%}')


/var/folders/7_/sz_ftshd1b1b9sppxlm9vj9m0000gn/T/ipykernel_33495/2016361176.py:27: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  F['expanded_30'] = ((F['first_exp_days'] >= 0) & (F['first_exp_days'] <= 30)).fillna(False).astype(int)
/var/folders/7_/sz_ftshd1b1b9sppxlm9vj9m0000gn/T/ipykernel_33495/2016361176.py:28: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  F['upgraded_30'] = F.index.isin(set(exp_ev.loc[exp_ev['kind'].eq('upgrade') & exp_ev['days_in'].between(0, 30), 'customer'])).astype(int)
/var/folders/7_/sz_ftshd1b1b9sppxl

base rates among monthly + observable 30d:
  n=16,091  renewal=50.9%  expand30=17.6%  upgrade30=10.1%  credits30=9.8%  both30=2.4%
post-M1 sample (renewed@30, tenure≥60d): n=5,610  renewed_60=97.3%


/var/folders/7_/sz_ftshd1b1b9sppxlm9vj9m0000gn/T/ipykernel_33495/2016361176.py:36: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  F['jobs_m2'] = g_m2.groupby('customer').size()
/var/folders/7_/sz_ftshd1b1b9sppxlm9vj9m0000gn/T/ipykernel_33495/2016361176.py:38: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  F['mom_growth'] = (F['jobs_m2'] - F['m1_jobs']) / F['m1_jobs'].replace(0, np.nan)
/var/folders/7_/sz_ftshd1b1b9sppxlm9vj9m0000gn/T/ipykernel_33495/2016361176.py:41: PerformanceWarning: DataFrame is highly fragmented.  This is usu

## 5.1 Bivariate tests

Each row is one product hypothesis. Continuous features: Mann–Whitney U on the two outcome groups. Categorical: χ². Effect = rank-biserial (MW) or Cramér's V (χ²).


In [21]:
def mw_test(df, feat, y, higher_means_better):
    a = df.loc[df[y]==1, feat].dropna()
    b = df.loc[df[y]==0, feat].dropna()
    if len(a) < 30 or len(b) < 30:
        return dict(test='MW', n1=len(a), n0=len(b), p=np.nan, effect=np.nan,
                    med1=np.nan, med0=np.nan, note='n<30')
    u, p = mannwhitneyu(a, b, alternative='two-sided')
    # rank-biserial: +1 if group 1 tends larger
    r = (2 * u) / (len(a) * len(b)) - 1
    direction = 'supports' if ((r > 0) == higher_means_better) and p < 0.05 else (
        'opposite' if p < 0.05 else 'ns')
    return dict(test='MW', n1=len(a), n0=len(b), p=p, effect=r,
                med1=float(a.median()), med0=float(b.median()), note=direction)

def chi_test(df, feat, y):
    tab = pd.crosstab(df[feat], df[y])
    if tab.shape[0] < 2 or tab.shape[1] < 2 or (tab.values < 5).any():
        return dict(test='chi2', n1=int((df[y]==1).sum()), n0=int((df[y]==0).sum()),
                    p=np.nan, effect=np.nan, med1=np.nan, med0=np.nan, note='sparse')
    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.values.sum()
    v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
    return dict(test='chi2', n1=int((df[y]==1).sum()), n0=int((df[y]==0).sum()),
                p=p, effect=v, med1=np.nan, med0=np.nan, note='sig' if p < 0.05 else 'ns')

def two_rate(df, mask, y, name):
    m = mask.fillna(False).astype(bool)
    if m.sum() < 30 or (~m).sum() < 30:
        return dict(test='rate', n1=int(m.sum()), n0=int((~m).sum()), p=np.nan, effect=np.nan,
                    med1=np.nan, med0=np.nan, note='n<30')
    p1, p0 = df.loc[m, y].mean(), df.loc[~m, y].mean()
    tab = pd.crosstab(m, df[y])
    chi2, p, _, _ = chi2_contingency(tab)
    n = tab.values.sum()
    v = np.sqrt(chi2 / (n * (min(tab.shape) - 1)))
    return dict(test='rate', n1=int(m.sum()), n0=int((~m).sum()), p=p, effect=v,
                med1=float(p1), med0=float(p0), note=f'{p1:.1%} vs {p0:.1%}')

# always attach flags on F (stale kernel may be missing them)
if 'platform' in F.columns:
    F['desktop'] = F['platform'].astype(str).str.lower().eq('desktop').astype(int)
if 'first_period' in F.columns:
    F['annual'] = (F['first_period'].astype(str) == 'year').astype(int)
if 'first_plan' in F.columns:
    F['plan_rank'] = F['first_plan'].map(RANK)
    F['rungs_above'] = 4 - F['plan_rank']
    F['can_upgrade'] = (F['rungs_above'].fillna(0) > 0).astype(int)

rows = []
R = F[(F['obs_30']==1) & (F['monthly']==1)].copy()          # main retention sample
R2 = F[(F['obs_60']==1) & (F['monthly']==1)].copy()         # month-2
E = F[(F['obs_30']==1) & (F['monthly']==1)].copy()          # expansion in 30d
E90 = F[(F['obs_90']==1) & (F['monthly']==1)].copy()        # later expansion

def _flag(df):
    if 'desktop' not in df.columns and 'platform' in df.columns:
        df['desktop'] = df['platform'].astype(str).str.lower().eq('desktop').astype(int)
    if 'annual' not in df.columns and 'first_period' in df.columns:
        df['annual'] = (df['first_period'].astype(str) == 'year').astype(int)
    if 'can_upgrade' not in df.columns and 'first_plan' in df.columns:
        df['can_upgrade'] = (df['first_plan'].map(RANK).fillna(4) < 4).astype(int)
    return df
R, R2, E, E90 = _flag(R), _flag(R2), _flag(E), _flag(E90)


# 1 Activation Velocity — faster first gen → higher 30d retention (H1 wrote M2; primary target is first renewal)
d = R[R['never_generated']==0]
t = mw_test(d, 'hours_to_first_gen', 'renewed_30', higher_means_better=False)
t.update(hypothesis='Activation Velocity', outcome='renewed_30', feature='hours_to_first_gen')
rows.append(t)
t = two_rate(d, d['hours_to_first_gen'] <= 24, 'renewed_30', 'gen_in_24h')
t.update(hypothesis='Activation Velocity (≤24h cut)', outcome='renewed_30', feature='gen_in_24h')
rows.append(t)

# 2 Early Stickiness
t = two_rate(R, R['w1_days'] >= 3, 'renewed_30', 'days7_ge3')
t.update(hypothesis='Early Stickiness (≥3 days in W1)', outcome='renewed_30', feature='w1_days>=3')
rows.append(t)
t = mw_test(R, 'w1_days', 'renewed_30', True)
t.update(hypothesis='Early Stickiness (continuous)', outcome='renewed_30', feature='w1_days')
rows.append(t)

# 3 Day-of-week cohort — Mon/Tue vs weekend signup
R['mon_tue'] = R['signup_dow'].isin([0, 1]).astype(int)
R['weekend_su'] = R['signup_weekend']
sub = R[R['signup_dow'].isin([0, 1, 5, 6])]
t = two_rate(sub, sub['mon_tue']==1, 'renewed_30', 'mon_tue')
t.update(hypothesis='Day-of-Week Cohort (MonTue vs weekend)', outcome='renewed_30', feature='signup_dow')
rows.append(t)

# 4 Binge vs pacing
d = R[R['w1_jobs'] > 0]
t = two_rate(d, d['binge_w1'] > 0.80, 'renewed_30', 'binge')
t.update(hypothesis='First-Day Binge vs Pacing (>80% W1 on one day)', outcome='renewed_30', feature='binge_w1>0.8')
rows.append(t)

# 5 Professional use case — weekday ratio vs expansion revenue
t = mw_test(E[E['never_generated']==0], 'm1_weekday', 'expanded_30', True)
t.update(hypothesis='Professional Use Case (weekday share)', outcome='expanded_30', feature='m1_weekday')
rows.append(t)
# also vs $
hi = E[E['never_generated']==0].copy()
hi['wd_hi'] = (hi['m1_weekday'] >= hi['m1_weekday'].median()).astype(int)
t = mw_test(hi, 'exp_revenue', 'wd_hi', True)
t.update(hypothesis='Professional Use Case ($ expansion)', outcome='exp_revenue', feature='weekday_high')
rows.append(t)

# 6 Working hours vs evening → upgrades (among those who can)
if 'can_upgrade' not in E.columns:
    E['can_upgrade'] = (E['first_plan'].map(RANK).fillna(4) < 4).astype(int)
U = E[E['can_upgrade']==1]
t = mw_test(U[U['never_generated']==0], 'm1_workhour', 'upgraded_30', True)
t.update(hypothesis='Working Hours vs Evening', outcome='upgraded_30', feature='m1_workhour')
rows.append(t)

# 7 Routine consistency — low hour_std → less churn
t = mw_test(R[R['m1_jobs']>=3], 'm1_hour_std', 'renewed_30', False)
t.update(hypothesis='Routine Consistency (hour std)', outcome='renewed_30', feature='m1_hour_std')
rows.append(t)

# 8 Activity cool-off — max_gap>=7 in month 1 → month 2 churn
t = two_rate(R2, R2['max_gap_28'] >= 7, 'renewed_60', 'gap')
t.update(hypothesis='Activity Cool-Off (max gap≥7d → M2 retain)', outcome='renewed_60', feature='max_gap_28>=7')
rows.append(t)

# 9 MoM growth → credits after day 60
d = E90[E90['m1_jobs']>0]
t = two_rate(d, d['mom_growth'] > 0.10, 'credits_after_60', 'mom')
t.update(hypothesis='MoM Growth (>10% M1→M2 → credits after 60d)', outcome='credits_after_60', feature='mom_growth>10%')
rows.append(t)

# 10 End-of-cycle spike → credits in the next cycle (days 30-60)
R2['credits_30_60'] = R2.index.isin(set(exp_ev.loc[exp_ev['kind'].eq('credits') & exp_ev['days_in'].between(30, 60), 'customer'])).astype(int)
t = mw_test(R2[R2['m1_jobs']>0], 'endcycle_spike', 'credits_30_60', True)
t.update(hypothesis='End-of-Cycle Spike → credits in days 30-60', outcome='credits_30_60', feature='endcycle_spike')
rows.append(t)

# 11 Volume-driven expansion
R2['hi_vol'] = (R2['m1_jobs'] >= R2['m1_jobs'].quantile(0.80)).astype(int)
R2['exp_m2'] = R2.index.isin(set(exp_ev.loc[exp_ev['days_in'].between(30, 60), 'customer'])).astype(int)
t = two_rate(R2, R2['hi_vol']==1, 'exp_m2', 'vol')
t.update(hypothesis='Volume-Driven Expansion (top 20% M1 jobs → exp days 30-60)', outcome='exp_m2', feature='m1_jobs_p80')
rows.append(t)

# 12 Platform superiority — retention
t = two_rate(R, R['desktop']==1, 'renewed_30', 'desk')
t.update(hypothesis='Platform Superiority (desktop vs mobile)', outcome='renewed_30', feature='desktop')
rows.append(t)

# 13 Platform expansion $
t = mw_test(E, 'exp_revenue', 'desktop', True)
t.update(hypothesis='Platform-Expansion (desktop vs $)', outcome='exp_revenue', feature='desktop')
rows.append(t)

# 14 Annual vs monthly — weekly gens by month 2 (jobs_m2/4)
# compare jobs_m2 among annual vs monthly, both obs_60
B = _flag(F[F['obs_60']==1].copy())
t = mw_test(B, 'jobs_m2', 'annual', True)
t.update(hypothesis='Annual vs Monthly (jobs in days 30-60)', outcome='jobs_m2', feature='annual')
rows.append(t)

# 15 Time to first expansion → second expansion
ex = F[(F['n_exp']>=1) & (F['obs_90']==1)].copy()
ex['early'] = (ex['first_exp_days'] <= 30).astype(int)
ex['second'] = (ex['n_exp'] >= 2).astype(int)
t = two_rate(ex, ex['early']==1, 'second', 'early')
t.update(hypothesis='Time to First Expansion (≤30d → 2nd event)', outcome='n_exp>=2', feature='first_exp_days<=30')
rows.append(t)

# 16 Regional tier adoption
tab = pd.crosstab(R['region'], R['first_plan'])
chi2, p, _, _ = chi2_contingency(tab)
n = tab.values.sum()
v = np.sqrt(chi2 / (n * (min(tab.shape)-1)))
rows.append(dict(hypothesis='Regional Tier Adoption (region × first_plan)',
                 outcome='first_plan mix', feature='region',
                 test='chi2', n1=len(R), n0=np.nan, p=p, effect=v,
                 med1=np.nan, med0=np.nan, note='sig' if p<0.05 else 'ns'))

# 17 Prompt evolution → upgrade
d = U[(U['w1_prompt_mean'].notna()) & (U['w4_prompt_mean'].notna())]
t = mw_test(d, 'prompt_evolution', 'upgraded_30', True)
t.update(hypothesis='Prompt Evolution (W4−W1 length → upgrade)', outcome='upgraded_30', feature='prompt_evolution')
rows.append(t)

# 18 Frustration / short prompts → LTV (revenue among obs_30)
t = mw_test(R[R['m1_jobs']>0], 'm1_short', 'renewed_30', False)
t.update(hypothesis='Frustration/Low-Effort (short-prompt share → retain)', outcome='renewed_30', feature='m1_short')
rows.append(t)
R['hi_short'] = (R['m1_short'] > 0.30).astype(int)
t = mw_test(R[R['m1_jobs']>0], 'revenue', 'hi_short', False)
t.update(hypothesis='Frustration/Low-Effort (short>30% → revenue)', outcome='revenue', feature='m1_short>0.3')
rows.append(t)

# 19 Complexity plateau — prompt std month1 vs month2 among obs_90
g_late = gen[(gen['days_since_pay']>30) & (gen['days_since_pay']<=90)]
std1 = g28.groupby('customer')['prompt_length'].std().rename('pstd_m1')
std2 = g_late.groupby('customer')['prompt_length'].std().rename('pstd_late')
P = E90.join(std1).join(std2)
dP = P[P['pstd_m1'].notna() & P['pstd_late'].notna()].copy()
t = two_rate(dP, dP['pstd_late'] < dP['pstd_m1'], 'renewed_60', 'plat')
t.update(hypothesis='Complexity Plateau (prompt std drops after M1 → M2 retain)',
         outcome='renewed_60', feature='prompt_std_drop')
rows.append(t)

# 20 Volume vs length — high jobs AND high prompt vs high jobs short prompt, on revenue
Q = R[R['m1_jobs']>0].copy()
Q['hi_jobs'] = Q['m1_jobs'] >= Q['m1_jobs'].quantile(0.75)
Q['hi_len'] = Q['m1_prompt_med'] >= Q['m1_prompt_med'].quantile(0.75)
both = Q[Q['hi_jobs']]
t = mw_test(both, 'revenue', 'hi_len', True)
t.update(hypothesis='Volume vs Length (high-vol: long vs short prompt → $)',
         outcome='revenue', feature='hi_prompt | hi_jobs')
rows.append(t)

res = pd.DataFrame(rows)
res['sig'] = res['p'] < 0.05
res = res[['hypothesis', 'outcome', 'feature', 'test', 'n1', 'n0', 'med1', 'med0', 'effect', 'p', 'note', 'sig']]
pd.set_option('display.max_colwidth', 80)
print(res.round(4).to_string(index=False))
print(f"\n{res['sig'].sum()} / {res['p'].notna().sum()} tests p<0.05")


                                                hypothesis          outcome             feature test    n1        n0   med1  med0  effect    p           note   sig
                                       Activation Velocity       renewed_30  hours_to_first_gen   MW  8131  7,792.00   0.01  0.02   -0.04 0.00       supports  True
                            Activation Velocity (≤24h cut)       renewed_30          gen_in_24h rate 15484    439.00   0.51  0.49    0.01 0.52 51.1% vs 49.4% False
                          Early Stickiness (≥3 days in W1)       renewed_30          w1_days>=3 rate  5336 10,755.00   0.55  0.49    0.06 0.00 55.2% vs 48.8%  True
                             Early Stickiness (continuous)       renewed_30             w1_days   MW  8194  7,897.00   2.00  2.00    0.07 0.00       supports  True
                    Day-of-Week Cohort (MonTue vs weekend)       renewed_30          signup_dow rate  4537  4,204.00   0.50  0.48    0.02 0.03 50.1% vs 47.7%  True
            Firs

Four targets (M1 vs post-M1; upgrade vs credits). Same tests, not one pooled y.


In [22]:
# same features, separate y — do not pool M1 with post-M1, or upgrade with credits
def pack(hyp, outcome, feat, d):
    d = dict(d)
    d.update(hypothesis=hyp, outcome=outcome, feature=feat)
    return d

grid = []
R_m1 = F[(F['obs_30']==1) & (F['monthly']==1)].copy()
R_post = F[(F['obs_60']==1) & (F['monthly']==1) & (F['renewed_30']==1)].copy()
if 'can_upgrade' in F.columns:
    U_up = F[(F['obs_30']==1) & (F['monthly']==1) & (F['can_upgrade']==1)].copy()
else:
    U_up = F[(F['obs_30']==1) & (F['monthly']==1)].copy()
E_cr = F[(F['obs_30']==1) & (F['monthly']==1)].copy()

targets = [
    ('M1 renewal', R_m1, 'renewed_30'),
    ('post-M1', R_post, 'renewed_60'),
    ('upgrade 30d', U_up, 'upgraded_30'),
    ('credits 30d', E_cr, 'credits_30'),
    ('both 30d', E_cr, 'both_30'),
]
mw_feats = [
    ('w1_days', True),
    ('m1_jobs', True),
    ('max_gap_28', False),
    ('hours_to_first_gen', False),
    ('binge_w1', False),
    ('w1_video', True),
    ('d14_video', True),
    ('m1_video', True),
    ('w1_video_image', True),
    ('d14_video_image', True),
    ('m1_video_image', True),
    ('w3_video_image', True),
    ('video_image_wow', True),
    ('w1_dow_med', True),
    ('d14_dow_med', True),
    ('m1_dow_med', True),
    ('wow_growth', True),
    ('endcycle_spike', True),
]
for tname, df, ycol in targets:
    if ycol not in df.columns or df[ycol].nunique() < 2 or len(df) < 80:
        print(f'skip {tname}: thin')
        continue
    print(f'\n=== {tname}  n={len(df):,}  base={df[ycol].mean():.1%} ===')
    for feat, higher in mw_feats:
        if feat not in df.columns:
            continue
        r = mw_test(df[df[feat].notna()], feat, ycol, higher)
        grid.append(pack(feat, tname, feat, r))

gdf = pd.DataFrame(grid)
print('\n', gdf[['outcome','feature','test','n1','n0','med1','med0','effect','p','note']].round(4).to_string(index=False))

piv = gdf.pivot_table(index='feature', columns='outcome', values='effect')
print('\neffect by target:\n', piv.round(3).to_string())
fig = px.imshow(piv, aspect='auto', color_continuous_scale='RdBu_r',
                color_continuous_midpoint=0, title='effect size by feature × target (do not pool)')
fig.show()



=== M1 renewal  n=16,091  base=50.9% ===

=== post-M1  n=5,610  base=97.3% ===



=== upgrade 30d  n=15,116  base=10.6% ===



=== credits 30d  n=16,091  base=9.8% ===

=== both 30d  n=16,091  base=2.4% ===



     outcome            feature test   n1    n0   med1  med0  effect    p     note
 M1 renewal            w1_days   MW 8194  7897   2.00  2.00    0.07 0.00 supports
 M1 renewal            m1_jobs   MW 8194  7897  20.00 16.00    0.10 0.00 supports
 M1 renewal         max_gap_28   MW 8131  7792  21.29 25.09   -0.14 0.00 supports
 M1 renewal hours_to_first_gen   MW 8131  7792   0.01  0.02   -0.04 0.00 supports
 M1 renewal           binge_w1   MW 8194  7897   0.77  0.83   -0.06 0.00 supports
 M1 renewal           w1_video   MW 8066  7750   1.00  1.00   -0.04 0.00 opposite
 M1 renewal          d14_video   MW 8108  7777   1.00  1.00   -0.06 0.00 opposite
 M1 renewal           m1_video   MW 8131  7792   1.00  1.00   -0.06 0.00 opposite
 M1 renewal     w1_video_image   MW 3373  2935   0.25  0.31   -0.03 0.04 opposite
 M1 renewal    d14_video_image   MW 3652  3089   0.31  0.35   -0.01 0.40       ns
 M1 renewal     m1_video_image   MW 3957  3287   0.35  0.38    0.01 0.71       ns
 M1 renewal   

In [23]:
# readable verdicts for the product table
ver = res.copy()
ver['verdict'] = np.where(ver['p'].isna(), 'not tested / thin n',
                  np.where(~ver['sig'], 'not supported',
                  np.where(ver['note']=='opposite', 'significant but opposite direction',
                           'supported')))
print(ver[['hypothesis', 'verdict', 'p', 'effect']].to_string(index=False))

fig = px.bar(res.dropna(subset=['p']).sort_values('p'),
             x='p', y='hypothesis', orientation='h', color='sig',
             title='hypothesis tests: p-values (log scale)')
fig.update_xaxes(type='log', title='p')
fig.update_layout(height=640)
fig.show()


                                                hypothesis                            verdict    p  effect
                                       Activation Velocity                          supported 0.00   -0.04
                            Activation Velocity (≤24h cut)                      not supported 0.52    0.01
                          Early Stickiness (≥3 days in W1)                          supported 0.00    0.06
                             Early Stickiness (continuous)                          supported 0.00    0.07
                    Day-of-Week Cohort (MonTue vs weekend)                          supported 0.03    0.02
            First-Day Binge vs Pacing (>80% W1 on one day)                          supported 0.00    0.05
                     Professional Use Case (weekday share) significant but opposite direction 0.00   -0.13
                       Professional Use Case ($ expansion) significant but opposite direction 0.00   -0.13
                                  Wor

## 5.2 ML — pooled + split targets, CatBoost + shallow forest

Same features (first 28d, no cohort). Pooled models sit next to the split ones for comparison.

| target | who is in the sample | y |
|---|---|---|
| pooled: M1 renewal | monthly, tenure ≥ 30d+grace | renewed_30 |
| post-M1 | those who already renewed at 30, tenure ≥ 60d+grace | renewed_60 |
| pooled: any expansion | monthly, tenure ≥ 30d+grace | expanded_30 |
| upgrade | monthly, can_upgrade, tenure ≥ 30d+grace | upgraded_30 |
| credits | monthly, tenure ≥ 30d+grace | credits_30 |
| both | monthly, tenure ≥ 30d+grace; skipped if thin | both_30 |


In [24]:
CAT = ['platform', 'region', 'first_plan']
NUM = [
    'hours_to_first_gen', 'days_to_first_gen', 'never_generated',
    'w1_days', 'd14_days', 'm1_days',
    'w1_jobs', 'd14_jobs', 'm1_jobs',
    'binge_w1', 'jobs_per_day_var', 'jobs_per_day_mean',
    'm1_weekday', 'weekday_weekend_ratio', 'm1_workhour',
    'm1_prompt_med', 'm1_prompt_max', 'm1_short', 'prompt_evolution',
    'wow_growth', 'cooloff_last10', 'recency_at_28', 'max_gap_28', 'endcycle_spike',
    'quota_burn_d15', 'frontload_d15', 'annual', 'days_signup_to_pay',
    'rungs_above', 'can_upgrade',
    'desktop', 'signup_dow', 'signup_month', 'hours_signup_to_pay',
    'm1_video', 'w1_video', 'd14_video', 'w3_video',
    'm1_video_image', 'w1_video_image', 'd14_video_image', 'w3_video_image',
    'video_image_wow',
    'm1_dow_med', 'w1_dow_med', 'd14_dow_med',
    'm1_fail', 'm1_hour_std',
]
NUM = [c for c in NUM if c in F.columns]
print('numeric features:', NUM)

def make_xy(df, ycol):
    d = df.copy()
    for c in CAT:
        if c in d.columns:
            d[c] = d[c].astype(str).fillna('na')
    cols = [c for c in CAT + NUM if c in d.columns]
    X = d[cols].replace([np.inf, -np.inf], np.nan)
    y = d[ycol].astype(int)
    return X, y, [c for c in CAT if c in X.columns], [c for c in NUM if c in X.columns]

def fit_two(X, y, label):
    cats = [c for c in CAT if c in X.columns]
    nums = [c for c in X.columns if c not in cats]
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=1, stratify=y)
    cb = CatBoostClassifier(iterations=600, depth=4, learning_rate=0.05, l2_leaf_reg=6,
                            eval_metric='AUC', random_seed=1, verbose=0,
                            early_stopping_rounds=50, use_best_model=True)
    cb.fit(Pool(X_tr, y_tr, cat_features=cats), eval_set=Pool(X_te, y_te, cat_features=cats))
    auc_cb = roc_auc_score(y_te, cb.predict_proba(X_te)[:, 1])
    Xf = X.copy()
    Xf[nums] = Xf[nums].fillna(Xf[nums].median())
    Xf = pd.get_dummies(Xf, columns=cats, drop_first=True)
    Xf_tr, Xf_te, yf_tr, yf_te = train_test_split(Xf, y, test_size=0.25, random_state=1, stratify=y)
    rf = RandomForestClassifier(n_estimators=250, max_depth=4, min_samples_leaf=80,
                                random_state=1, n_jobs=-1)
    rf.fit(Xf_tr, yf_tr)
    auc_rf = roc_auc_score(yf_te, rf.predict_proba(Xf_te)[:, 1])
    print(f'{label}: n={len(y):,}  base={y.mean():.1%}  CatBoost AUC {auc_cb:.3f}  RF AUC {auc_rf:.3f}')
    return dict(label=label, cb=cb, rf=rf, X_tr=X_tr, X_te=X_te, y_tr=y_tr, y_te=y_te,
                Xf_te=Xf_te, yf_te=yf_te, cats=cats, auc_cb=auc_cb, auc_rf=auc_rf)

base_m = F[(F['obs_30']==1) & (F['monthly']==1)]
post_m = F[(F['obs_60']==1) & (F['monthly']==1) & (F['renewed_30']==1)]
up_m = F[(F['obs_30']==1) & (F['monthly']==1) & (F['can_upgrade']==1)]

jobs = {}
X, y, _, _ = make_xy(base_m, 'renewed_30')
jobs['m1'] = fit_two(X, y, 'pooled: M1 renewal')
X, y, _, _ = make_xy(post_m, 'renewed_60')
jobs['post'] = fit_two(X, y, 'post-M1 (among M1 retained)')
X, y, _, _ = make_xy(base_m, 'expanded_30')
jobs['exp'] = fit_two(X, y, 'pooled: any expansion 30d')
X, y, _, _ = make_xy(up_m, 'upgraded_30')
jobs['up'] = fit_two(X, y, 'upgrade 30d')
X, y, _, _ = make_xy(base_m, 'credits_30')
jobs['cr'] = fit_two(X, y, 'credits 30d')
if 'both_30' in base_m.columns and base_m['both_30'].sum() >= 80 and base_m['both_30'].nunique() == 2:
    X, y, _, _ = make_xy(base_m, 'both_30')
    jobs['both'] = fit_two(X, y, 'both upgrade+credits 30d')

rows = []
for k, j in jobs.items():
    yall = pd.concat([j['y_tr'], j['y_te']])
    rows.append(dict(key=k, model=j['label'], n=len(yall), base=float(yall.mean()),
                     auc_cb=j['auc_cb'], auc_rf=j['auc_rf']))
print('\nAUC comparison (pooled vs split):\n', pd.DataFrame(rows).round(3).to_string(index=False))


numeric features: ['hours_to_first_gen', 'days_to_first_gen', 'never_generated', 'w1_days', 'd14_days', 'm1_days', 'w1_jobs', 'd14_jobs', 'm1_jobs', 'binge_w1', 'jobs_per_day_var', 'jobs_per_day_mean', 'm1_weekday', 'weekday_weekend_ratio', 'm1_workhour', 'm1_prompt_med', 'm1_prompt_max', 'm1_short', 'prompt_evolution', 'wow_growth', 'cooloff_last10', 'recency_at_28', 'max_gap_28', 'endcycle_spike', 'quota_burn_d15', 'frontload_d15', 'annual', 'days_signup_to_pay', 'rungs_above', 'can_upgrade', 'desktop', 'signup_dow', 'signup_month', 'hours_signup_to_pay', 'm1_video', 'w1_video', 'd14_video', 'w3_video', 'm1_video_image', 'w1_video_image', 'd14_video_image', 'w3_video_image', 'video_image_wow', 'm1_dow_med', 'w1_dow_med', 'd14_dow_med', 'm1_fail', 'm1_hour_std']


pooled: M1 renewal: n=16,091  base=50.9%  CatBoost AUC 0.653  RF AUC 0.642


post-M1 (among M1 retained): n=5,610  base=97.3%  CatBoost AUC 0.837  RF AUC 0.847


pooled: any expansion 30d: n=16,091  base=17.6%  CatBoost AUC 0.898  RF AUC 0.874


upgrade 30d: n=15,116  base=10.6%  CatBoost AUC 0.868  RF AUC 0.855


credits 30d: n=16,091  base=9.8%  CatBoost AUC 0.951  RF AUC 0.937


both upgrade+credits 30d: n=16,091  base=2.4%  CatBoost AUC 0.922  RF AUC 0.904

AUC comparison (pooled vs split):
  key                       model     n  base  auc_cb  auc_rf
  m1          pooled: M1 renewal 16091  0.51    0.65    0.64
post post-M1 (among M1 retained)  5610  0.97    0.84    0.85
 exp   pooled: any expansion 30d 16091  0.18    0.90    0.87
  up                 upgrade 30d 15116  0.11    0.87    0.85
  cr                 credits 30d 16091  0.10    0.95    0.94
both    both upgrade+credits 30d 16091  0.02    0.92    0.90


In [25]:
# SHAP per target — show more than top-10 so video/image, dow, gaps stay visible
TOP_SHAP = 25
shap_imp = {}
for key, job in jobs.items():
    sv = shap.TreeExplainer(job['cb']).shap_values(
        Pool(job['X_te'], job['y_te'], cat_features=job['cats']))
    imp = (pd.DataFrame({'feature': job['X_te'].columns,
                         'mean_abs_shap': np.abs(sv).mean(axis=0)})
           .sort_values('mean_abs_shap', ascending=False))
    shap_imp[key] = imp
    show = imp.head(TOP_SHAP)
    print(f"\n{job['label']} SHAP (top {len(show)}/{len(imp)}):\n", show.round(4).to_string(index=False))
    fig = px.bar(show.sort_values('mean_abs_shap'), x='mean_abs_shap', y='feature',
                 orientation='h', title=f"CatBoost SHAP — {job['label']}")
    fig.update_layout(height=max(520, 22 * len(show) + 80))
    fig.show()

# compare rankings: pooled vs split
def rank_corr(a, b, la, lb):
    cmp = (a.set_index('feature')['mean_abs_shap'].rename(la)
           .to_frame().join(b.set_index('feature')['mean_abs_shap'].rename(lb))).dropna()
    cmp = cmp / cmp.sum()
    rho, p = spearmanr(cmp[la], cmp[lb])
    print(f'SHAP rank corr {la} vs {lb}: rho={rho:.3f}, p={p:.4f}')

rank_corr(shap_imp['m1'], shap_imp['post'], 'M1 pooled', 'post-M1')
rank_corr(shap_imp['exp'], shap_imp['up'], 'exp pooled', 'upgrade')
rank_corr(shap_imp['exp'], shap_imp['cr'], 'exp pooled', 'credits')
rank_corr(shap_imp['up'], shap_imp['cr'], 'upgrade', 'credits')
rank_corr(shap_imp['m1'], shap_imp['exp'], 'M1 pooled', 'exp pooled')
if 'both' in shap_imp:
    rank_corr(shap_imp['exp'], shap_imp['both'], 'exp pooled', 'both')

# one heatmap: relative SHAP across models (all features that appear in any top-25)
frames = []
for k, imp in shap_imp.items():
    s = imp.set_index('feature')['mean_abs_shap']
    s = s / s.sum()
    frames.append(s.rename(jobs[k]['label']))
heat = pd.concat(frames, axis=1).fillna(0)
keep = heat.max(axis=1).sort_values(ascending=False).head(25).index
heat = heat.loc[keep]
print('\nrelative SHAP (share of total) — top features across models:')
print(heat.round(3).to_string())
fig = px.imshow(heat, aspect='auto', color_continuous_scale='YlOrRd',
                title='relative SHAP share by feature × model (pooled vs split)')
fig.update_layout(height=max(560, 22 * len(heat) + 120), margin=dict(l=180))
fig.show()



pooled: M1 renewal SHAP (top 25/51):
               feature  mean_abs_shap
               region           0.27
          rungs_above           0.12
       quota_burn_d15           0.07
       cooloff_last10           0.07
        recency_at_28           0.06
   hours_to_first_gen           0.06
    days_to_first_gen           0.05
           m1_weekday           0.04
         signup_month           0.04
           max_gap_28           0.03
              w1_jobs           0.03
       w3_video_image           0.03
        m1_prompt_med           0.03
          m1_hour_std           0.03
              m1_days           0.02
           m1_dow_med           0.02
           first_plan           0.02
   days_signup_to_pay           0.02
       endcycle_spike           0.02
weekday_weekend_ratio           0.02
        frontload_d15           0.02
  hours_signup_to_pay           0.02
              m1_jobs           0.02
    jobs_per_day_mean           0.02
             d14_days           0.01


post-M1 (among M1 retained) SHAP (top 25/51):


               feature  mean_abs_shap
          rungs_above           0.63
        recency_at_28           0.15
               region           0.09
           max_gap_28           0.09
              w1_days           0.08
             w3_video           0.07
             binge_w1           0.07
           m1_weekday           0.07
          m1_hour_std           0.05
    jobs_per_day_mean           0.05
     jobs_per_day_var           0.05
             d14_jobs           0.05
              m1_jobs           0.04
        m1_prompt_max           0.04
         signup_month           0.04
          m1_workhour           0.03
   hours_to_first_gen           0.03
             m1_video           0.03
           wow_growth           0.03
              w1_jobs           0.03
             d14_days           0.03
          can_upgrade           0.02
        frontload_d15           0.02
              m1_days           0.02
weekday_weekend_ratio           0.02



pooled: any expansion 30d SHAP (top 25/51):
             feature  mean_abs_shap
        rungs_above           0.38
            m1_jobs           0.36
           m1_video           0.21
            m1_days           0.17
         first_plan           0.15
     quota_burn_d15           0.12
  jobs_per_day_mean           0.12
        can_upgrade           0.11
   jobs_per_day_var           0.11
     cooloff_last10           0.11
           d14_jobs           0.10
      frontload_d15           0.08
  days_to_first_gen           0.07
          d14_video           0.06
hours_signup_to_pay           0.06
             region           0.06
            w1_jobs           0.06
       signup_month           0.06
 days_signup_to_pay           0.05
         m1_weekday           0.05
           m1_short           0.04
      m1_prompt_max           0.04
        m1_hour_std           0.04
 hours_to_first_gen           0.04
      recency_at_28           0.04



upgrade 30d SHAP (top 25/51):
             feature  mean_abs_shap
        rungs_above           0.41
         first_plan           0.35
            m1_jobs           0.12
hours_signup_to_pay           0.10
  jobs_per_day_mean           0.10
            m1_days           0.07
   jobs_per_day_var           0.07
           m1_video           0.07
          d14_video           0.07
           m1_short           0.07
      recency_at_28           0.06
  days_to_first_gen           0.06
     cooloff_last10           0.05
 days_signup_to_pay           0.05
             region           0.05
 hours_to_first_gen           0.04
            w1_jobs           0.04
     m1_video_image           0.04
           d14_days           0.04
         max_gap_28           0.04
           w3_video           0.04
           d14_jobs           0.04
     endcycle_spike           0.03
            w1_days           0.03
    d14_video_image           0.03



credits 30d SHAP (top 25/51):
           feature  mean_abs_shap
          m1_jobs           0.48
      rungs_above           0.33
   quota_burn_d15           0.26
         m1_video           0.24
          m1_days           0.19
         d14_jobs           0.19
    frontload_d15           0.13
      can_upgrade           0.13
          w1_jobs           0.12
   cooloff_last10           0.10
       first_plan           0.10
 jobs_per_day_var           0.09
      m1_hour_std           0.09
         w1_video           0.08
       max_gap_28           0.08
    recency_at_28           0.07
     signup_month           0.07
          w1_days           0.07
         w3_video           0.06
        d14_video           0.05
    m1_prompt_max           0.05
jobs_per_day_mean           0.04
         m1_short           0.04
   w3_video_image           0.03
       wow_growth           0.03



both upgrade+credits 30d SHAP (top 25/51):
               feature  mean_abs_shap
          rungs_above           0.48
              m1_jobs           0.24
     jobs_per_day_var           0.13
             m1_video           0.13
            d14_video           0.11
              m1_days           0.11
          m1_hour_std           0.11
       m1_video_image           0.11
             m1_short           0.10
    jobs_per_day_mean           0.10
             d14_days           0.09
       cooloff_last10           0.09
           max_gap_28           0.09
              w1_days           0.09
             binge_w1           0.09
   days_signup_to_pay           0.07
        frontload_d15           0.07
             w3_video           0.06
             d14_jobs           0.05
              w1_jobs           0.05
weekday_weekend_ratio           0.05
           wow_growth           0.05
  hours_signup_to_pay           0.04
       quota_burn_d15           0.04
          m1_workhour         

SHAP rank corr M1 pooled vs post-M1: rho=0.281, p=0.0457
SHAP rank corr exp pooled vs upgrade: rho=0.727, p=0.0000
SHAP rank corr exp pooled vs credits: rho=0.725, p=0.0000
SHAP rank corr upgrade vs credits: rho=0.525, p=0.0001


SHAP rank corr M1 pooled vs exp pooled: rho=0.407, p=0.0030
SHAP rank corr exp pooled vs both: rho=0.655, p=0.0000

relative SHAP (share of total) — top features across models:
                     pooled: M1 renewal  post-M1 (among M1 retained)  pooled: any expansion 30d  upgrade 30d  credits 30d  both upgrade+credits 30d
feature                                                                                                                                            
rungs_above                        0.09                         0.30                       0.12         0.18         0.10                      0.16
region                             0.20                         0.04                       0.02         0.02         0.00                      0.01
first_plan                         0.02                         0.01                       0.05         0.15         0.03                      0.01
m1_jobs                            0.01                         0.02               

In [26]:
TOP_PERM = 20

def perm_importance_df(proba_fn, X, y, n_repeats=8, seed=1):
    rng = np.random.default_rng(seed)
    baseline = roc_auc_score(y, proba_fn(X))
    rec = []
    for col in X.columns:
        drops = []
        for _ in range(n_repeats):
            Xp = X.copy()
            Xp[col] = rng.permutation(Xp[col].to_numpy())
            drops.append(baseline - roc_auc_score(y, proba_fn(Xp)))
        rec.append((col, float(np.mean(drops)), float(np.std(drops, ddof=1))))
    out = pd.DataFrame(rec, columns=['feature', 'perm_mean', 'perm_std'])
    out['ci95_lo'] = out['perm_mean'] - 1.96 * out['perm_std']
    out['ci95_hi'] = out['perm_mean'] + 1.96 * out['perm_std']
    return out.sort_values('perm_mean', ascending=False)

perm_imp = {}
for key, job in jobs.items():
    pcb = perm_importance_df(lambda Z, m=job: m['cb'].predict_proba(Z)[:, 1],
                             job['X_te'], job['y_te'], n_repeats=8)
    perm_imp[key] = pcb
    show = pcb.head(TOP_PERM)
    print(f"\n{job['label']} perm ΔAUC ±95% CI (top {len(show)}/{len(pcb)}):\n",
          show.round(4).to_string(index=False))
    top = show.sort_values('perm_mean')
    fig = go.Figure()
    fig.add_bar(y=top['feature'], x=top['perm_mean'], orientation='h',
                error_x=dict(type='data', array=1.96*top['perm_std']),
                marker_color=PALETTE[0])
    fig.update_layout(title=f"permutation ΔAUC — {job['label']}",
                      height=max(520, 22 * len(top) + 80))
    fig.show()



pooled: M1 renewal perm ΔAUC ±95% CI (top 20/51):
               feature  perm_mean  perm_std  ci95_lo  ci95_hi
               region       0.04      0.01     0.03     0.05
          rungs_above       0.01      0.00     0.01     0.01
         signup_month       0.01      0.00     0.00     0.01
   hours_to_first_gen       0.00      0.00     0.00     0.01
       quota_burn_d15       0.00      0.00     0.00     0.01
    days_to_first_gen       0.00      0.00    -0.00     0.01
        recency_at_28       0.00      0.00    -0.00     0.01
       cooloff_last10       0.00      0.00     0.00     0.01
           max_gap_28       0.00      0.00    -0.00     0.00
        m1_prompt_med       0.00      0.00     0.00     0.00
weekday_weekend_ratio       0.00      0.00    -0.00     0.00
   days_signup_to_pay       0.00      0.00     0.00     0.00
           first_plan       0.00      0.00    -0.00     0.00
        frontload_d15       0.00      0.00     0.00     0.00
       w1_video_image       0.00 


post-M1 (among M1 retained) perm ΔAUC ±95% CI (top 20/51):
           feature  perm_mean  perm_std  ci95_lo  ci95_hi
      rungs_above       0.12      0.03     0.07     0.17
          w1_days       0.01      0.00    -0.00     0.02
       m1_weekday       0.01      0.01    -0.00     0.02
jobs_per_day_mean       0.01      0.00     0.00     0.01
      can_upgrade       0.00      0.00    -0.00     0.01
          m1_jobs       0.00      0.00    -0.00     0.01
       max_gap_28       0.00      0.00    -0.00     0.01
 jobs_per_day_var       0.00      0.00    -0.00     0.01
  d14_video_image       0.00      0.00     0.00     0.01
         d14_jobs       0.00      0.00    -0.01     0.01
           region       0.00      0.00    -0.01     0.01
    recency_at_28       0.00      0.01    -0.01     0.01
   endcycle_spike       0.00      0.00     0.00     0.00
   quota_burn_d15       0.00      0.00    -0.00     0.00
         w1_video       0.00      0.00    -0.00     0.00
         m1_video       0.0


pooled: any expansion 30d perm ΔAUC ±95% CI (top 20/51):
            feature  perm_mean  perm_std  ci95_lo  ci95_hi
       rungs_above       0.03      0.00     0.02     0.03
           m1_jobs       0.02      0.00     0.02     0.03
          m1_video       0.02      0.00     0.01     0.02
    quota_burn_d15       0.01      0.00     0.01     0.02
        first_plan       0.01      0.00     0.00     0.01
      signup_month       0.00      0.00     0.00     0.01
           m1_days       0.00      0.00     0.00     0.01
 days_to_first_gen       0.00      0.00     0.00     0.00
       can_upgrade       0.00      0.00     0.00     0.01
         d14_video       0.00      0.00     0.00     0.00
hours_to_first_gen       0.00      0.00     0.00     0.00
          m1_short       0.00      0.00     0.00     0.00
          d14_jobs       0.00      0.00    -0.00     0.00
            region       0.00      0.00     0.00     0.00
 jobs_per_day_mean       0.00      0.00     0.00     0.00
     frontloa


upgrade 30d perm ΔAUC ±95% CI (top 20/51):
             feature  perm_mean  perm_std  ci95_lo  ci95_hi
        rungs_above       0.03      0.01     0.02     0.04
         first_plan       0.02      0.00     0.01     0.03
            m1_jobs       0.00      0.00     0.00     0.01
  days_to_first_gen       0.00      0.00     0.00     0.01
          d14_video       0.00      0.00     0.00     0.01
 hours_to_first_gen       0.00      0.00     0.00     0.00
hours_signup_to_pay       0.00      0.00     0.00     0.00
           m1_short       0.00      0.00     0.00     0.00
           m1_video       0.00      0.00     0.00     0.00
     m1_video_image       0.00      0.00     0.00     0.00
  jobs_per_day_mean       0.00      0.00     0.00     0.00
     cooloff_last10       0.00      0.00    -0.00     0.00
            m1_days       0.00      0.00    -0.00     0.00
             region       0.00      0.00     0.00     0.00
       signup_month       0.00      0.00     0.00     0.00
     quota_


credits 30d perm ΔAUC ±95% CI (top 20/51):
            feature  perm_mean  perm_std  ci95_lo  ci95_hi
       rungs_above       0.03      0.00     0.03     0.03
           m1_jobs       0.02      0.00     0.01     0.02
    quota_burn_d15       0.01      0.00     0.01     0.02
          m1_video       0.01      0.00     0.01     0.01
      signup_month       0.00      0.00     0.00     0.00
       can_upgrade       0.00      0.00     0.00     0.00
          w1_video       0.00      0.00     0.00     0.00
        first_plan       0.00      0.00     0.00     0.00
         d14_video       0.00      0.00     0.00     0.00
          d14_jobs       0.00      0.00    -0.00     0.00
     frontload_d15       0.00      0.00    -0.00     0.00
    w3_video_image       0.00      0.00     0.00     0.00
days_signup_to_pay       0.00      0.00     0.00     0.00
           m1_days       0.00      0.00    -0.00     0.00
    cooloff_last10       0.00      0.00    -0.00     0.00
 jobs_per_day_mean       0.


both upgrade+credits 30d perm ΔAUC ±95% CI (top 20/51):
            feature  perm_mean  perm_std  ci95_lo  ci95_hi
       rungs_above       0.02      0.01     0.01     0.03
           m1_jobs       0.01      0.00     0.00     0.02
          m1_video       0.01      0.00     0.00     0.01
         d14_video       0.01      0.00     0.00     0.01
    cooloff_last10       0.00      0.00     0.00     0.01
       can_upgrade       0.00      0.00     0.00     0.00
     frontload_d15       0.00      0.00     0.00     0.00
 jobs_per_day_mean       0.00      0.00    -0.00     0.00
           m1_days       0.00      0.00    -0.00     0.00
          w1_video       0.00      0.00     0.00     0.00
    endcycle_spike       0.00      0.00     0.00     0.00
          m1_short       0.00      0.00    -0.00     0.00
          w3_video       0.00      0.00    -0.00     0.00
        m1_weekday       0.00      0.00    -0.00     0.00
days_signup_to_pay       0.00      0.00    -0.00     0.00
       d14_dow

In [27]:
# leftover: M1 model objects under jobs['m1'] for anything downstream
cb, X_te, y_te = jobs['m1']['cb'], jobs['m1']['X_te'], jobs['m1']['y_te']
imp = shap_imp['m1']
print('downstream aliases set: cb, X_te, y_te, imp  ← M1 renewal')


downstream aliases set: cb, X_te, y_te, imp  ← M1 renewal


## 5.3 Behavioural segmentation

Paying customers, tenure ≥ 30d + 7d grace — **monthly and annual**, same usage rules. Metrics from generations in days 0–28. First matching rule.

Never started is ~1% — folded into **one-shot**. `max_gap ≥ 7` does not split (recency to day 28 is baked in) — not a bucket.

Annual has no 30-day subscription renewal; paid renewal is reported on monthly only. ARPU is shown as cash (year prepaid lands in days 0–30) and as a monthly equivalent.


Why `w1_days` ≥ 3 (not 2 or 4). Week-1 active days are 0–7. One-shot is already 0–1 and gone. The remaining cut is “did a week-1 habit start.” Plot the distribution and the jump in expansion / paid renewal at each extra day — keep 3 only if that is where the curve bends.


In [28]:
# why 3 active days in week 1 — distribution + outcomes, then pick the cut
B = F[F['obs_30']==1].copy()
B['w1_days'] = B['w1_days'].fillna(0)
B['w1_jobs'] = B['w1_jobs'].fillna(0)
B['m1_jobs'] = B['m1_jobs'].fillna(0)
B['jobs_after_w1'] = (B['m1_jobs'] - B['w1_jobs']).clip(lower=0)
B['used_after_w1'] = (B['jobs_after_w1'] > 0).astype(int)
B['period'] = np.where(B['first_period'].astype(str).eq('year'), 'annual', 'monthly')
B['w1'] = B['w1_days'].clip(0, 7).round().astype(int)

print(f'30d-observable base: n={len(B):,}  monthly={(B["period"]=="monthly").sum():,}  '
      f'annual={(B["period"]=="annual").sum():,}')
print('\nw1_days distribution (all periods)')
print(B['w1'].value_counts().sort_index().to_string())
print('\n% of base:', (B['w1'].value_counts(normalize=True).sort_index()*100).round(1).to_string())

by = B.groupby('w1').agg(
    n=('w1', 'size'),
    expand_30=('expanded_30', 'mean'),
    used_after_w1=('used_after_w1', 'mean'),
    m1_jobs=('m1_jobs', 'median'),
).copy()
mon = B[B['period']=='monthly']
by['paid_renewal'] = mon.groupby('w1')['renewed_30'].mean()
by['n_monthly'] = mon.groupby('w1').size()
by['share'] = by['n'] / by['n'].sum()
by['d_expand'] = by['expand_30'].diff()
by['d_renew'] = by['paid_renewal'].diff()
print('\noutcome by exact week-1 active days  (paid_renewal = monthly only)')
print(by.round(3).to_string())

print('\ncut comparison  (habit = w1_days ≥ k)')
rows = []
for k in [2, 3, 4]:
    hab = B['w1'] >= k
    rows.append({
        'cut': f'≥{k} days',
        'n_habit': int(hab.sum()),
        'share_habit': hab.mean(),
        'expand_habit': B.loc[hab, 'expanded_30'].mean(),
        'expand_below': B.loc[~hab, 'expanded_30'].mean(),
        'd_expand_pp': B.loc[hab, 'expanded_30'].mean() - B.loc[~hab, 'expanded_30'].mean(),
        'renew_habit': mon.loc[mon['w1']>=k, 'renewed_30'].mean(),
        'renew_below': mon.loc[mon['w1']<k, 'renewed_30'].mean(),
        'd_renew_pp': mon.loc[mon['w1']>=k, 'renewed_30'].mean() - mon.loc[mon['w1']<k, 'renewed_30'].mean(),
    })
cuts = pd.DataFrame(rows)
print(cuts.round(3).to_string(index=False))
best_exp = cuts.loc[cuts['d_expand_pp'].idxmax(), 'cut']
best_ren = cuts.loc[cuts['d_renew_pp'].idxmax(), 'cut']
print(f'\nlargest expansion gap: {best_exp}   largest monthly-renewal gap: {best_ren}')
print('binary gap grows as the cut moves into the tail (>=4 looks better because habit is smaller and more selected).')
print('distribution: 67% of the base is 0-2 days. 0-1 is flat on expansion (~7%). First step is day 2, then a slope.')
print('>=2 would call 59% of the base habit. >=4 dumps w1=3 (n~2.6k, expand 22%) into weak week-1.')
print('keep >=3: after the 1-2 pile, first repeated return, habit is a minority (33%).')

hist = B.groupby(['w1', 'period']).size().rename('n').reset_index()
fig = px.bar(hist, x='w1', y='n', color='period', barmode='stack',
             title='week-1 active days — 30d-observable (monthly + annual)')
fig.add_vline(x=2.5, line_dash='dash', annotation_text='habit cut (≥3)')
fig.update_xaxes(title='distinct days with a generation in week 1', dtick=1)
fig.update_layout(yaxis_title='customers')
fig.show()
try:
    fig.write_image('w1_days_dist.png')
except Exception as e:
    print('png skip', type(e).__name__)

out = by.reset_index().melt(id_vars='w1', value_vars=['expand_30', 'paid_renewal', 'used_after_w1'],
                            var_name='metric', value_name='rate')
fig = px.line(out, x='w1', y='rate', color='metric', markers=True,
              title='outcomes by week-1 active days  (paid renewal = monthly only)')
fig.add_vline(x=2.5, line_dash='dash', annotation_text='habit cut')
fig.update_xaxes(title='w1_days', dtick=1)
fig.update_yaxes(tickformat='.0%')
fig.show()
try:
    fig.write_image('w1_days_cut.png')
except Exception as e:
    print('png skip', type(e).__name__)

fig = px.bar(by.reset_index(), x='w1', y='d_expand',
             title='marginal expansion gain from one extra week-1 day')
fig.add_vline(x=2.5, line_dash='dash')
fig.update_xaxes(title='w1_days', dtick=1)
fig.update_yaxes(tickformat='.0%', title='Δ expand_30 vs previous day')
fig.show()


30d-observable base: n=18,048  monthly=16,093  annual=1,955

w1_days distribution (all periods)
w1
0     376
1    7080
2    4587
3    2584
4    1565
5     929
6     496
7     431

% of base: w1
0    2.10
1   39.20
2   25.40
3   14.30
4    8.70
5    5.10
6    2.70
7    2.40

outcome by exact week-1 active days  (paid_renewal = monthly only)
       n  expand_30  used_after_w1  m1_jobs  paid_renewal  n_monthly  share  d_expand  d_renew
w1                                                                                            
0    376       0.06           0.39     0.00          0.47        275   0.02       NaN      NaN
1   7080       0.07           0.26     7.00          0.48       6374   0.39      0.01     0.02
2   4587       0.14           0.45    17.00          0.49       4107   0.25      0.07     0.01
3   2584       0.22           0.60    32.00          0.52       2321   0.14      0.08     0.02
4   1565       0.32           0.73    65.00          0.55       1391   0.09      0.11   

png skip ValueError


png skip ValueError


In [29]:
# five segments — summary §10
S = F[F['obs_30']==1].copy()
S['period'] = np.where(S['first_period'].astype(str).eq('year'), 'annual', 'monthly')
S['monthly'] = (S['period']=='monthly').astype(int)
S['jobs_after_w1'] = (S['m1_jobs'] - S['w1_jobs']).clip(lower=0)

# usage percentile on this base (rank of m1_jobs)
S['usage_pp'] = S['m1_jobs'].rank(method='average', pct=True)

# tail from the Lorenz curve: smallest top-of-base that holds ≥50% of jobs
j = S['m1_jobs'].sort_values(ascending=False)
_tot = float(j.sum()); job_share = j.cumsum() / (_tot if _tot else np.nan)
user_share = np.arange(1, len(j)+1) / len(j)
k50 = int(np.searchsorted(job_share.to_numpy(), 0.50, side='left')) + 1
p_lorenz = 1 - k50 / len(S)
# not thinner than p95, not a handful of accounts
TAIL_P = float(np.clip(p_lorenz, 0.95, 0.99))
p80 = S['m1_jobs'].quantile(0.80)
jobs_tail = S['m1_jobs'].quantile(TAIL_P)

print('usage distribution (m1_jobs)')
print(S['m1_jobs'].describe(percentiles=[.5,.8,.9,.95,.99]).round(1).to_string())
print(f'\nLorenz: top {100*(1-p_lorenz):.1f}% of customers hold 50% of jobs → raw p={p_lorenz:.3f}')
print(f'tail cut TAIL_P={TAIL_P:.3f}  (floor 0.95)  jobs ≥ {jobs_tail:.0f}')
for top in [0.10, 0.05, 0.01]:
    thr = S['m1_jobs'].quantile(1-top)
    mass = S.loc[S['m1_jobs']>=thr, 'm1_jobs'].sum() / S['m1_jobs'].sum()
    print(f'  top {100*top:.0f}%  (≥{thr:.0f} jobs)  hold {100*mass:.0f}% of jobs')

lor = pd.DataFrame({'user_share': user_share, 'job_share': job_share.to_numpy()})
fig = px.line(lor, x='user_share', y='job_share', title='Lorenz: cumulative jobs vs customers (sorted by usage)')
fig.add_vline(x=1-TAIL_P, line_dash='dash', annotation_text=f'tail p={TAIL_P:.2f}')
fig.add_hline(y=0.5, line_dash='dot')
fig.update_xaxes(tickformat='.0%', title='share of customers (heaviest first)')
fig.update_yaxes(tickformat='.0%', title='share of jobs')
fig.show()
fig = px.histogram(S.loc[S['m1_jobs']>0], x='m1_jobs', log_x=True, nbins=60, title='m1_jobs (log x, zeros dropped)')
fig.add_vline(x=p80, line_dash='dash', annotation_text=f'p80={p80:.0f}')
fig.add_vline(x=jobs_tail, line_dash='dash', annotation_text=f'p{int(TAIL_P*100)}={jobs_tail:.0f}')
fig.show()

oneshot = (S['w1_days'] <= 1) & (S['jobs_after_w1'] == 0)
active = S['w1_days'] >= 3
power = active & (S['usage_pp'] > TAIL_P)
S['segment'] = np.select(
    [oneshot,
     S['w1_days'] < 3,
     power,
     S['m1_jobs'] >= p80],
    ['1. one-shot',
     '2. weak week-1',
     '5. active, top usage',
     '4. habit, heavy'],
    default='3. habit, light',
)
order = ['1. one-shot', '2. weak week-1', '3. habit, light', '4. habit, heavy', '5. active, top usage']
print(f'\np80 of m1_jobs = {p80:.0f}  |  active + usage_pp > {TAIL_P:.2f}')
print(S['segment'].value_counts().reindex(order).to_string())
print('\n% of base:\n', (S['segment'].value_counts(normalize=True).reindex(order)*100).round(1).to_string())

sizes = S['segment'].value_counts().reindex(order).rename('n').reset_index()
sizes.columns = ['segment', 'n']
sizes['share'] = sizes['n'] / sizes['n'].sum()
sizes['label'] = sizes.apply(lambda r: f"{int(r['n']):,} ({100*r['share']:.1f}%)", axis=1)
fig = px.bar(sizes, x='segment', y='n', text='label', title='segment size')
fig.update_traces(textposition='outside')
fig.update_layout(yaxis_title='customers')
fig.show()
try:
    fig.write_image('share_of_base.png')
except Exception:
    pass
mix = S.groupby(['segment','period']).size().rename('n').reset_index()
mix['segment'] = pd.Categorical(mix['segment'], categories=order, ordered=True)
print('\nmix monthly vs annual:\n', S.groupby('segment')['period'].value_counts().unstack(fill_value=0).reindex(order).to_string())
fig = px.bar(mix, x='segment', y='n', color='period', barmode='stack',
             title='segment size: monthly vs annual')
fig.show()
fig = px.pie(sizes, names='segment', values='n', title='share of 30d-observable base (monthly + annual)')
fig.update_traces(textinfo='percent+value+label')
fig.show()

prof = S.groupby('segment').agg(
    n=('renewed_30', 'size'),
    w1_days=('w1_days', 'median'),
    m1_days=('m1_days', 'median'),
    m1_jobs=('m1_jobs', 'median'),
    after_w1=('jobs_after_w1', 'median'),
    binge=('binge_w1', 'median'),
    hrs_to_gen=('hours_to_first_gen', 'median'),
    m1_video=('m1_video', 'median'),
    expand_30=('expanded_30', 'mean'),
    upgrade_30=('upgraded_30', 'mean'),
    credits_30=('credits_30', 'mean'),
    desktop=('desktop', 'mean'),
    revenue=('revenue', 'median'),
).reindex(order)
prof['share'] = S['segment'].value_counts(normalize=True).reindex(prof.index)
prof['share_annual'] = S.groupby('segment')['period'].apply(lambda s: (s=='annual').mean()).reindex(prof.index)
prof['renewal'] = S[S['period']=='monthly'].groupby('segment')['renewed_30'].mean().reindex(prof.index)
print('\n', prof.round(3).to_string())

long = prof.reset_index().melt(id_vars='segment',
                               value_vars=['renewal', 'expand_30', 'upgrade_30', 'credits_30'],
                               var_name='metric', value_name='rate')
fig = px.bar(long, x='segment', y='rate', color='metric', barmode='group',
             title='renewal (monthly) and expansion by segment')
fig.update_yaxes(tickformat='.0%')
fig.show()
try:
    fig.write_image('renewal_exp_by_segment.png')
except Exception:
    pass

med = prof.reset_index().melt(id_vars='segment',
                              value_vars=['w1_days', 'm1_jobs', 'after_w1'],
                              var_name='metric', value_name='median')
fig = px.bar(med, x='segment', y='median', color='metric', barmode='group',
             title='defining metrics (median)')
fig.show()


usage distribution (m1_jobs)
count   18,048.00
mean       107.80
std        462.30
min          0.00
50%         17.00
80%         82.00
90%        200.00
95%        428.60
99%      1,566.50
max     17,055.00

Lorenz: top 2.9% of customers hold 50% of jobs → raw p=0.971
tail cut TAIL_P=0.971  (floor 0.95)  jobs ≥ 719
  top 10%  (≥200 jobs)  hold 74% of jobs
  top 5%  (≥429 jobs)  hold 61% of jobs
  top 1%  (≥1567 jobs)  hold 32% of jobs



p80 of m1_jobs = 82  |  active + usage_pp > 0.97
segment
1. one-shot             5487
2. weak week-1          6556
3. habit, light         3378
4. habit, heavy         2162
5. active, top usage     465

% of base:
 segment
1. one-shot            30.40
2. weak week-1         36.30
3. habit, light        18.70
4. habit, heavy        12.00
5. active, top usage    2.60



mix monthly vs annual:
 period                annual  monthly
segment                              
1. one-shot              559     4928
2. weak week-1           728     5828
3. habit, light          333     3045
4. habit, heavy          268     1894
5. active, top usage      67      398



                          n  w1_days  m1_days  m1_jobs  after_w1  binge  hrs_to_gen  m1_video  expand_30  upgrade_30  credits_30  desktop  revenue  share  share_annual  renewal
segment                                                                                                                                                                        
1. one-shot           5487     1.00     1.00     5.00      0.00   1.00        0.01      1.00       0.04        0.03        0.01     0.50    28.44   0.30          0.10     0.46
2. weak week-1        6556     2.00     2.00    17.00      2.00   0.80        0.03      1.00       0.14        0.08        0.07     0.70    42.89   0.36          0.11     0.51
3. habit, light       3378     3.00     5.00    27.00      2.00   0.50        0.02      1.00       0.20        0.10        0.11     0.70    56.00   0.19          0.10     0.50
4. habit, heavy       2162     4.00     9.00   185.00     66.00   0.49        0.12      0.17       0.46        0.23   

In [30]:
# overlays on 4 (expansion) and 5 (margin tail)
hv = S[S['segment']=='4. habit, heavy']
pw = S[S['segment']=='5. active, top usage']
print(f'habit, heavy: n={len(hv):,}  p80={p80:.0f}')
print(f'active, top usage: n={len(pw):,}  TAIL_P={TAIL_P:.3f}  jobs≥{jobs_tail:.0f}')
if len(hv) > 50:
    by_plan = hv.groupby('first_plan').agg(
        n=('expanded_30','size'),
        renewal=('renewed_30','mean'),
        upgrade=('upgraded_30','mean'),
        credits=('credits_30','mean'),
        jobs=('m1_jobs','median'),
        video=('m1_video','median'),
    ).reindex(LADDER).dropna(how='all')
    print('\nby entry plan:\n', by_plan.round(3).to_string())
    fig = px.bar(by_plan.reset_index(), x='first_plan', y=['upgrade', 'credits'],
                 barmode='group', title='4. habit, heavy: upgrade vs credits by entry plan')
    fig.update_yaxes(tickformat='.0%')
    fig.show()

if len(pw) > 30:
    by_plan5 = pw.groupby('first_plan').agg(
        n=('expanded_30','size'),
        renewal=('renewed_30','mean'),
        upgrade=('upgraded_30','mean'),
        credits=('credits_30','mean'),
        jobs=('m1_jobs','median'),
        video=('m1_video','median'),
    ).reindex(LADDER).dropna(how='all')
    print('\n5. active, top usage by entry plan:\n', by_plan5.round(3).to_string())
    fig = px.bar(by_plan5.reset_index(), x='first_plan', y=['upgrade', 'credits'],
                 barmode='group', title='5. active, top usage: upgrade vs credits by entry plan')
    fig.update_yaxes(tickformat='.0%')
    fig.show()

print('\nperiod mix in 4 and 5')
print(S.loc[S['segment'].isin(['4. habit, heavy','5. active, top usage'])]
      .groupby('segment')['period'].value_counts().unstack(fill_value=0).to_string())

print('\n1 one-shot            → return after week 1')
print('2 weak week-1         → w1_days → 3')
print('3 habit, light        → paid first renewal; do not max jobs')
print('4 habit, heavy        → expansion pool; pitch by plan')
print('5 active, top usage   → margin tail: jobs per $ and whether they paid more')


habit, heavy: n=2,162  p80=82
active, top usage: n=465  TAIL_P=0.971  jobs≥719

by entry plan:
                        n  renewal  upgrade  credits   jobs  video
first_plan                                                       
Higgsfield Basic     332     0.42     0.04     0.16 168.00   0.01
Higgsfield Pro       932     0.56     0.24     0.20 176.50   0.13
Higgsfield Ultimate  554     0.66     0.47     0.25 208.50   0.41
Higgsfield Creator    23     0.70     0.43     0.35 187.00   0.53



5. active, top usage by entry plan:
                        n  renewal  upgrade  credits     jobs  video
first_plan                                                         
Higgsfield Basic      14     0.64     0.21     0.71 1,318.00   0.00
Higgsfield Pro       140     0.71     0.54     0.36 1,129.00   0.01
Higgsfield Ultimate  214     0.76     0.61     0.40 1,376.50   0.06
Higgsfield Creator    10     0.90     0.70     0.30 1,263.50   0.07



period mix in 4 and 5
period                annual  monthly
segment                              
4. habit, heavy          268     1894
5. active, top usage      67      398

1 one-shot            → return after week 1
2 weak week-1         → w1_days → 3
3 habit, light        → paid first renewal; do not max jobs
4 habit, heavy        → expansion pool; pitch by plan
5 active, top usage   → margin tail: jobs per $ and whether they paid more


## 5.4 Metrics by segment


Compare segments on value, paid retention, usage, expansion, and a margin proxy (jobs per $).


In [31]:
# 30d revenue from charges (fair ARPU). lifetime S['revenue'] mixes tenure.
S = S.drop(columns=[c for c in ['rev_30','rev_credits_30','rev_sub_30',
                                'jobs_per_usd','used_after_w1','used_m2'] if c in S.columns],
           errors='ignore')
chm = charges_df[['customer', 'revenue_time', 'sales_amount']].copy()
chm['is_credits'] = charges_df['is_credits'] if 'is_credits' in charges_df.columns \
    else charges_df['subscription_plan'].eq('Credits Package')
chm = chm.merge(S[['first_pay']], left_on='customer', right_index=True, how='inner')
chm['days'] = (chm['revenue_time'] - chm['first_pay']).dt.total_seconds() / 86400
ch30 = chm[chm['days'].between(0, 30)]
rev30 = ch30.groupby('customer')['sales_amount'].sum().rename('rev_30')
rev_cr = ch30.loc[ch30['is_credits']].groupby('customer')['sales_amount'].sum().rename('rev_credits_30')
rev_sub = ch30.loc[~ch30['is_credits']].groupby('customer')['sales_amount'].sum().rename('rev_sub_30')
S = S.join(rev30).join(rev_cr).join(rev_sub)
for c in ['rev_30', 'rev_credits_30', 'rev_sub_30']:
    S[c] = S[c].fillna(0)
# annual prepaid is a year of $ on day 0 — cash ARPU is real, meq is comparable across periods
S['rev_sub_meq'] = np.where(S['period'].eq('annual'), S['rev_sub_30'] / 12.0, S['rev_sub_30'])
S['rev_30_meq'] = S['rev_sub_meq'] + S['rev_credits_30']
S['jobs_per_usd'] = S['m1_jobs'] / S['rev_30_meq'].replace(0, np.nan)
S['used_after_w1'] = (S['jobs_after_w1'] > 0).astype(int)
S['used_m2'] = np.nan
if 'jobs_m2' in S.columns and 'obs_60' in S.columns:
    S.loc[S['obs_60']==1, 'used_m2'] = (S.loc[S['obs_60']==1, 'jobs_m2'] > 0).astype(float)

order = ['1. one-shot', '2. weak week-1', '3. habit, light', '4. habit, heavy', '5. active, top usage']
g = S.groupby('segment', observed=True)

paid_ren = S[S['period']=='monthly'].groupby('segment')['renewed_30'].mean()
n_ann = S.groupby('segment')['period'].apply(lambda s: (s=='annual').sum())
M = pd.DataFrame({
    'n': g.size(),
    'n_annual': n_ann,
    'share': g.size() / len(S),
    'share_annual': n_ann / g.size(),
    'ARPU_30d': g['rev_30'].mean(),
    'ARPU_meq': g['rev_30_meq'].mean(),
    'ARPU_sub_30d': g['rev_sub_30'].mean(),
    'ARPU_credits_30d': g['rev_credits_30'].mean(),
    'exp_ARPU': g['exp_revenue'].mean() if 'exp_revenue' in S.columns else np.nan,
    'paid_renewal': paid_ren,
    'used_after_w1': g['used_after_w1'].mean(),
    'used_m2': g['used_m2'].mean(),
    'w1_days': g['w1_days'].median(),
    'm1_days': g['m1_days'].median(),
    'm1_jobs': g['m1_jobs'].median(),
    'jobs_after_w1': g['jobs_after_w1'].median(),
    'expand_30': g['expanded_30'].mean(),
    'upgrade_30': g['upgraded_30'].mean(),
    'credits_30': g['credits_30'].mean(),
    'm1_video': g['m1_video'].median(),
    'jobs_per_usd': g['jobs_per_usd'].median(),
    'desktop': g['desktop'].mean(),
}).reindex(order)
print('metrics by segment\n')
print(M.round(3).to_string())

idx_cols = ['ARPU_meq', 'paid_renewal', 'used_after_w1', 'w1_days', 'm1_jobs',
            'jobs_after_w1', 'expand_30', 'm1_video', 'jobs_per_usd']
idx = M[idx_cols].copy()
for c in idx_cols:
    mu = idx[c].mean()
    idx[c] = idx[c] / mu if mu else np.nan
print('\nindexed to mean of the four segments (=1). >1 = high for this bucket\n')
print(idx.round(2).to_string())

def bar(df, cols, title, pct=False):
    cols = [c for c in cols if c in df.columns and df[c].notna().any()]
    if not cols:
        return
    long = df.reset_index().melt(id_vars='segment', value_vars=cols,
                                 var_name='metric', value_name='value')
    fig = px.bar(long, x='segment', y='value', color='metric', barmode='group', title=title)
    if pct:
        fig.update_yaxes(tickformat='.0%')
    fig.show()

bar(M, ['ARPU_30d', 'ARPU_meq', 'ARPU_credits_30d'], 'ARPU in first 30 days ($): cash vs monthly-equivalent')
bar(M, ['share_annual'], 'share annual in the bucket', pct=True)
bar(M, ['paid_renewal', 'used_after_w1', 'used_m2'],
    'retention: paid renewal (monthly only) vs still generating', pct=True)
bar(M, ['w1_days', 'm1_days'], 'activity: active days (median)')
bar(M, ['m1_jobs', 'jobs_after_w1'], 'activity: jobs (median)')
bar(M, ['expand_30', 'upgrade_30', 'credits_30'], 'expansion in 30d', pct=True)
bar(M, ['jobs_per_usd'], 'margin proxy: month-1 jobs per $ (higher = more compute per dollar)')
bar(M, ['m1_video'], 'video share (median)', pct=True)

fig = px.imshow(idx, aspect='auto', color_continuous_midpoint=1,
                color_continuous_scale='RdBu_r',
                title='each metric ÷ mean across segments  (what this bucket is about)')
fig.update_layout(height=420)
fig.show()

# key metric: product lever that is also distinctive (index far from 1)
levers = {
    '1. one-shot':      ['used_after_w1', 'w1_days', 'paid_renewal'],
    '2. weak week-1':   ['w1_days', 'used_after_w1', 'paid_renewal'],
    '3. habit, light':  ['paid_renewal', 'expand_30', 'm1_jobs'],
    '4. habit, heavy':  ['expand_30', 'ARPU_credits_30d', 'jobs_per_usd', 'm1_video'],
    '5. active, top usage': ['jobs_per_usd', 'expand_30', 'm1_jobs', 'm1_video'],
}
print('\n--- suggested key metric ---')
print(f'{"segment":22s}  {"watch":20s}  {"index":>6s}  note')
notes = {
    '1. one-shot':     'paid renewal is auto-charge (~50% everywhere). Watch return after week 1 (0 here by construction).',
    '2. weak week-1':  'they already came back a bit. Watch week-1 active days → 3.',
    '3. habit, light': 'jobs are not the goal (guardrail). Watch paid renewal on monthly; annual has no 30d renew.',
    '4. habit, heavy': 'renewal already high. Watch expansion $ and jobs per $ (margin).',
    '5. active, top usage': 'the compute tail. Watch jobs per $ and whether they expanded; not a retention KPI.',
}
for seg in order:
    cols = [c for c in levers[seg] if c in idx.columns]
    row = idx.loc[seg, cols].astype(float)
    pick = (row - 1).abs().idxmax()
    print(f'{seg:22s}  {pick:20s}  {row[pick]:6.2f}  {notes[seg]}')


metrics by segment

                         n  n_annual  share  share_annual  ARPU_30d  ARPU_meq  ARPU_sub_30d  ARPU_credits_30d  exp_ARPU  paid_renewal  used_after_w1  used_m2  w1_days  m1_days  m1_jobs  jobs_after_w1  expand_30  upgrade_30  credits_30  m1_video  jobs_per_usd  desktop
segment                                                                                                                                                                                                                                                                    
1. one-shot           5487       559   0.30          0.10     44.59     22.19         44.38              0.21      4.28          0.46           0.00     0.10     1.00     1.00     5.00           0.00       0.04        0.03        0.01      1.00          0.29     0.50
2. weak week-1        6556       728   0.36          0.11     59.67     30.01         57.69              1.99     13.80          0.51           0.61     0.37     2.00     2.00 


--- suggested key metric ---
segment                 watch                  index  note
1. one-shot             used_after_w1           0.00  paid renewal is auto-charge (~50% everywhere). Watch return after week 1 (0 here by construction).
2. weak week-1          w1_days                 0.62  they already came back a bit. Watch week-1 active days → 3.
3. habit, light         m1_jobs                 0.09  jobs are not the goal (guardrail). Watch paid renewal on monthly; annual has no 30d renew.
4. habit, heavy         m1_video                0.27  renewal already high. Watch expansion $ and jobs per $ (margin).
5. active, top usage    m1_jobs                 4.21  the compute tail. Watch jobs per $ and whether they expanded; not a retention KPI.
